# EVDetect — MLflow Experiment Tracking

This notebook loads the finalized, saved models (produced by the multiclass and binary supervised-learning notebooks) and logs their functional and non-functional metrics to MLflow for centralized experiment tracking and comparison.

**Requires:** trained model artifacts present in a local `models/` folder, and `EVSE-B-PowerCombined.csv` in a local `data/` folder (see repository README).


In [ ]:
# --- Path configuration (edit if your folder layout differs) ---
MODELS_DIR = 'models/'   # folder containing saved .pkl / .json / .keras model files
DATA_PATH = 'data/EVSE-B-PowerCombined.csv'


In [1]:
# Import the necessary libraries and functions
import os
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.tensorflow
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from mlflow.tracking import MlflowClient
import json
import warnings
import xgboost as xgb
import inspect
import tempfile
from mlflow.models.signature import infer_signature

2025-10-24 17:15:17.506561: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Set the tracking URI:

In [2]:
# Set the tracking uri
mlflow.set_tracking_uri('http://127.0.0.1:5000')

See all the active and deleted experiments:

In [ ]:
# Initialize MLflow client
client = MlflowClient()

# Get all experiments (including deleted ones)
all_experiments = client.search_experiments(view_type = mlflow.entities.ViewType.ALL)

print('All MLflow Experiments:')
print('=' * 80)

if not all_experiments:
    print('No experiments found')
else:
    for exp in all_experiments:
        status_emoji = '✅' if exp.lifecycle_stage == 'active' else '🗑️'
        print(f'{status_emoji} ID: {exp.experiment_id}')
        print(f'   Name: {exp.name}')
        print(f'   Status: {exp.lifecycle_stage}')

        # Count runs in this experiment
        try:
            runs = client.search_runs(
                experiment_ids = [exp.experiment_id],
                run_view_type = mlflow.entities.ViewType.ALL
            )
            print(f'   Runs: {len(runs)} total')
        except Exception:
            print(f'   Runs: Unable to count')
        print('-' * 40)

Permanently delete or restore any experiments and their runs:

In [5]:
# Initialize MLflow client
client = MlflowClient()

def completely_delete_experiment(experiment_name):
    """
    Completely delete an MLflow experiment and all its runs.
    This action cannot be undone.
    """
    try:
        # Get experiment by name
        experiment = client.get_experiment_by_name(experiment_name)
        
        if experiment is None:
            print(f"No experiment found with name: '{experiment_name}'")
            return False
            
        experiment_id = experiment.experiment_id
        print(f"Found experiment: '{experiment_name}' (ID: {experiment_id})")
        
        # Get all runs in this experiment (including deleted ones)
        runs = client.search_runs(
            experiment_ids = [experiment_id],
            run_view_type = mlflow.entities.ViewType.ALL  # Include active, deleted, and archived
        )
        
        print(f'Found {len(runs)} runs in this experiment')
        
        # Delete all runs first
        for run in runs:
            run_id = run.info.run_id
            try:
                if run.info.lifecycle_stage != 'deleted':
                    print(f"  Deleting run: {run_id}")
                    client.delete_run(run_id)
                else:
                    print(f'  Run already deleted: {run_id}')
            except Exception as e:
                print(f'  Warning: Could not delete run {run_id}: {e}')

        # Now delete the experiment
        if experiment.lifecycle_stage == 'active':
            print('Step 1: Moving experiment to deleted state...')
            client.delete_experiment(experiment_id)

        print('Step 2: Permanently deleting experiment...')
        client.delete_experiment(experiment_id)  # Second call for permanent deletion
        
        print(f"✅ Experiment '{experiment_name}' and all its runs have been permanently deleted")
        return True
        
    except Exception as e:
        print(f'❌ Error deleting experiment: {e}')
        return False

# Execute the deletion (commented out for safety - uncomment to run)
# completely_delete_experiment('Multiclass Classification (Charging State)')
# completely_delete_experiment('Multiclass Classification (Both States)')
# completely_delete_experiment('Binary Classification (Charging State)')
# completely_delete_experiment('Binary Classification (Both States)')

Create or get an existing active experiment:

In [37]:
# Initialize MLflow client
client = MlflowClient()

# Create or get experiment - handles deleted experiments with same name
experiment_name = 'Binary Classification (Both States)'

def create_or_get_experiment(name):
    """
    Create a new experiment or get existing active one.
    If a deleted experiment with the same name exists, restore it.
    """
    experiment = client.get_experiment_by_name(name)
    if experiment:
        if experiment.lifecycle_stage == 'active':
            print(f"Using existing active experiment: '{name}' (ID: {experiment.experiment_id})")
            return experiment.experiment_id
        else:
            print(f'Experiment exists but is not active (lifecycle_stage = {experiment.lifecycle_stage}). Attempting to restore...')
            client.restore_experiment(experiment.experiment_id)
            print(f"Restored experiment: '{name}' (ID: {experiment.experiment_id})")
            return experiment.experiment_id
    else:
        experiment_id = mlflow.create_experiment(name)
        print(f"Created new experiment: '{name}' (ID: {experiment_id})")
        return experiment_id

# Create or get the experiment and set it as active
experiment_id = create_or_get_experiment(experiment_name)
mlflow.set_experiment(experiment_name)
print(f"Active experiment: '{experiment_name}' (ID: {experiment_id})")

Created new experiment: 'Binary Classification (Both States)' (ID: 4)
Active experiment: 'Binary Classification (Both States)' (ID: 4)


# Multiclass Classification (Charging State)

In [3]:
# Set the experiment as active
mlflow.set_experiment('Multiclass Classification (Charging State)')

<Experiment: artifact_location='/Users/yannis/mlflow-artifacts/1', creation_time=1761295456430, experiment_id='1', last_update_time=1761297798102, lifecycle_stage='active', name='Multiclass Classification (Charging State)', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
# Load the sheet into a DataFrame
df = pd.read_csv(DATA_PATH)

# Display basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115298 entries, 0 to 115297
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   time           115298 non-null  object 
 1   shunt_voltage  115298 non-null  int64  
 2   bus_voltage_V  115298 non-null  float64
 3   current_mA     115298 non-null  int64  
 4   power_mW       115298 non-null  int64  
 5   State          115298 non-null  object 
 6   Attack         115298 non-null  object 
 7   Attack-Group   115298 non-null  object 
 8   Label          115298 non-null  object 
 9   interface      115298 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 8.8+ MB


In [5]:
# Print the first rows of the DataFrame
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp


In [6]:
# Define mapping dictionary
attack_mapping = {
    'syn-flood': 'DoS',
    'tcp-flood': 'DoS',
    'cryptojacking': 'Cryptojacking',
    'vuln-scan': 'Recon',
    'syn-stealth': 'Recon',
    'Backdoor': 'Backdoor',
    'none': 'Benign'
}

# Apply mapping to create 'Attack-New' column
df['Attack-New'] = df['Attack'].map(attack_mapping)

# Print the first rows of the DataFrame to verify the new column
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp,DoS
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp,DoS
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp,DoS
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp,DoS
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp,DoS


In [7]:
# Filter data for 'charging' state
df_charging = df[df['State'] == 'charging'].copy()

# Print the first few rows of the filtered DataFrame
df_charging.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New
34884,12/26/2023 15:17,660,5.185,652,3380,charging,none,none,benign,none,Benign
34885,12/26/2023 15:17,769,5.189,675,3500,charging,none,none,benign,none,Benign
34886,12/26/2023 15:17,525,5.197,512,2660,charging,none,none,benign,none,Benign
34887,12/26/2023 15:17,690,5.181,642,3340,charging,none,none,benign,none,Benign
34888,12/26/2023 15:17,638,5.189,574,2780,charging,none,none,benign,none,Benign


In [8]:
# Calculate the number of instances for each unique value in 'Attack-New' column
df.loc[df['State'] == 'charging', 'Attack-New'].value_counts()

Attack-New
Backdoor         14172
Cryptojacking     8474
Benign            6947
Name: count, dtype: int64

In [9]:
# Select numerical features for classification (excluding categorical columns)
numerical_features = ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']

# Define features (X) and target variable (Y)
X = df_charging[numerical_features]
Y = df_charging['Attack-New']

In [10]:
# Split the dataset into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42, stratify = Y, shuffle = True)

In [11]:
# Split the training set into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.2, random_state = 42, stratify = Y_train, shuffle = True)

In [12]:
# Set the encoder 
label_encoder = LabelEncoder()

# Encode 'Attack-New' labels as numbers for each subset
# Note: This is done after splitting to avoid data leakage
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_val_encoded = label_encoder.transform(Y_val)
Y_test_encoded = label_encoder.transform(Y_test)

# Check what each number represents
print('Label encoder classes:', label_encoder.classes_)

Label encoder classes: ['Backdoor' 'Benign' 'Cryptojacking']


In [13]:
# Create the scaler
scaler = StandardScaler()

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Shape of the scaled data
print(f'Shape of X_train_scaled: {X_train_scaled.shape}')
print(f'Shape of X_val_scaled: {X_val_scaled.shape}')
print(f'Shape of X_test_scaled: {X_test_scaled.shape}')

Shape of X_train_scaled: (18939, 4)
Shape of X_val_scaled: (4735, 4)
Shape of X_test_scaled: (5919, 4)


In [ ]:
# Suppress specific warnings for cleaner output
warnings.filterwarnings('ignore', category = UserWarning)

# Models list
models_to_log = [
    ('KNN', MODELS_DIR + 'knn_multi_charging_best_model.pkl', 'sklearn', 'scaled', False),
    ('KNN SMOTE', MODELS_DIR + 'knn_multi_charging_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('RBF SVM', MODELS_DIR + 'rbf_svm_multi_charging_best_model.pkl', 'sklearn', 'scaled', False),
    ('RBF SVM SMOTE', MODELS_DIR + 'rbf_svm_multi_charging_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('XGBoost', MODELS_DIR + 'xgboost_multi_charging_best_model.json', 'xgboost', 'scaled', False),
    ('XGBoost SMOTE', MODELS_DIR + 'xgboost_multi_charging_smote_best_model.json', 'xgboost', 'scaled', True),
    ('Logistic Regression', MODELS_DIR + 'logistic_regression_multi_charging_best_model.pkl', 'sklearn', 'scaled', False),
    ('Logistic Regression SMOTE', MODELS_DIR + 'logistic_regression_multi_charging_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('Random Forest', MODELS_DIR + 'random_forest_multi_charging_best_model.pkl', 'sklearn', 'original', False),
    ('Random Forest SMOTE', MODELS_DIR + 'random_forest_multi_charging_smote_best_model.pkl', 'sklearn', 'original', True),
    ('ANN', MODELS_DIR + 'ann_multi_charging_model.keras', 'keras', 'scaled', False),
    ('ANN SMOTE', MODELS_DIR + 'ann_multi_charging_smote_model.keras', 'keras', 'scaled', True),
    ('Stacking Classifier', MODELS_DIR + 'stacking_clf_multi_charging_best_model.pkl', 'sklearn', 'original', False)
]

# Load production results CSV
# Clean CSV columns for MLflow logging
df_prod = pd.read_csv('df_production_results_multi_charging.csv')

# Ensure numeric
for col in ['Prediction Time (sec)','F1 Macro', 'Memory Used (kB)', 'CPU Time (sec)', 'Physical Cores', 'Logical Cores', 'CPU % Used (Raw)',
            'CPU % Used (Normalized)','Model Size (MB)']:
    df_prod[col] = pd.to_numeric(df_prod[col], errors = 'coerce')

# Sanity checks
required_globals = ['X_train', 'X_test', 'X_test_scaled', 'Y_test_encoded', 'label_encoder']
missing = [g for g in required_globals if g not in globals()]
if missing:
    raise RuntimeError(f'Please ensure these globals exist before running: {missing}')

# Helper functions
def ensure_dataframe(X, ref_columns):
    if isinstance(X, np.ndarray):
        return pd.DataFrame(X, columns = list(ref_columns))
    if isinstance(X, pd.DataFrame):
        if list(X.columns) != list(ref_columns) and X.shape[1] == len(ref_columns):
            return pd.DataFrame(X.values, columns = list(ref_columns))
        return X
    raise ValueError('X must be numpy array or pandas DataFrame')

def model_includes_scaler(model):
    if isinstance(model, Pipeline):
        for step in model.named_steps.values():
            cls_name = step.__class__.__name__.lower()
            if 'scaler' in cls_name or 'standardscaler' in cls_name or 'minmax' in cls_name or 'preprocessor' in cls_name:
                return True
    return False

def _safe_params_dict(params: dict, prefix = ''):
    safe = {}
    for k, v in params.items():
        key = f'{prefix}{k}'
        if isinstance(v, (str, int, float, bool)) or v is None:
            safe[key] = v
        else:
            try:
                s = json.dumps(v) if isinstance(v, (dict, list)) else str(v)
                safe[key] = s[:200]
            except Exception:
                safe[key] = str(type(v))
    return safe


def keras_params_summary(model: keras.Model):
    params = {}
    try:
        params['keras_total_params'] = model.count_params()
    except Exception:
        params['keras_total_params'] = None
    params['keras_layers_count'] = len(model.layers)
    for i, layer in enumerate(model.layers):
        try:
            cfg = layer.get_config()
        except Exception:
            cfg = {}
        params[f'layer_{i}_class'] = layer.__class__.__name__
        if 'units' in cfg:
            params[f'layer_{i}_units'] = cfg.get('units')
        if 'activation' in cfg:
            params[f'layer_{i}_activation'] = cfg.get('activation')
    try:
        opt = model.optimizer.get_config()
        params['optimizer'] = model.optimizer.__class__.__name__
        for k, v in opt.items():
            params[f'opt_{k}'] = v if isinstance(v, (str, int, float, bool)) else str(v)
    except Exception:
        pass
    try:
        params['loss'] = model.loss if not callable(model.loss) else str(model.loss)
    except Exception:
        pass
    return params

def extract_xgb_from_pipeline(m):
    if not isinstance(m, Pipeline):
        return None
    for step in m.named_steps.values():
        cls = step.__class__.__name__.lower()
        if 'xgb' in cls or hasattr(step, 'get_xgb_params'):
            return step
    return None

def safe_head_example(X, n = 8):
    try:
        if isinstance(X, pd.DataFrame):
            return X.head(n)
        if isinstance(X, np.ndarray):
            return pd.DataFrame(X[:n], columns = X_train.columns)
    except Exception:
        pass
    return None

def safe_predict_for_signature(model, X_example, framework):
    try:
        fw = (framework or '').lower()
        if fw == 'keras':
            preds = model.predict(X_example, verbose = 0)
            return preds
        elif fw == 'xgboost':
            if isinstance(model, xgb.core.Booster):
                d = xgb.DMatrix(X_example)
                preds = model.predict(d)
                return preds
            else:
                if hasattr(model, 'predict'):
                    return model.predict(X_example)
        else:
            return model.predict(X_example)
    except Exception:
        return None

def safe_log_params_iter(obj, prefix = ''):
    safe = {}
    try:
        gp = obj.get_params()
        for k, v in gp.items():
            try:
                if isinstance(v, (list, dict)):
                    safe[f'{prefix}{k}'] = json.dumps(v)[:400]
                elif hasattr(v, '__name__') and getattr(v, '__name__'):
                    safe[f'{prefix}{k}'] = getattr(v, '__name__')
                else:
                    safe[f'{prefix}{k}'] = v
            except Exception:
                safe[f'{prefix}{k}'] = str(type(v))
    except Exception:
        pass
    return safe

def mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = None):
    fw = (framework or '').lower()
    example_df = safe_head_example(X_example) if X_example is not None else None

    input_example_np = None
    signature = None
    input_example = None

    if example_df is not None:
        input_example_np = example_df.values if isinstance(example_df, pd.DataFrame) else example_df
        input_example = input_example_np
        preds = safe_predict_for_signature(model, input_example_np, fw)
        try:
            signature = infer_signature(input_example_np, preds) if preds is not None else infer_signature(input_example_np, None)
        except Exception:
            signature = None

    with tempfile.TemporaryDirectory(prefix = 'mlflow_model_') as tmp_path:
        try:
            if fw == 'keras':
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                # Skip passing input_example here to avoid MLflow directory error
                try:
                    mlflow.keras.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception as e:
                    print('  -> Primary keras log_model failed:', e)
                    saved_path = os.path.join(tmp_path, 'model.keras')
                    os.makedirs(saved_path, exist_ok = True)
                    try:
                        model.save(saved_path, include_optimizer = True)
                    except Exception:
                        model.save(saved_path)
                    mlflow.log_artifact(saved_path, artifact_path = artifact_name)
                    return True

            elif fw == 'xgboost':
                inner = extract_xgb_from_pipeline(model) if isinstance(model, Pipeline) else None
                candidate = inner or model
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.xgboost.log_model(xgb_model=candidate, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                try:
                    if isinstance(candidate, xgb.core.Booster):
                        p = os.path.join(tmp_path, 'booster.json')
                        candidate.save_model(p)
                        mlflow.log_artifact(p, artifact_path = artifact_name)
                        return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

            else:
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.sklearn.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

        except Exception as e:
            print(f'mlflow_log_model_auto: primary path exception: {e}')
            return False

# Unified prediction/proba extractor (works for pipelines, sklearn, keras, xgboost)
def get_predictions_and_probas(model, framework, X_eval_df, label_encoder):
    if framework == 'keras':
        y_pred_proba = model.predict(X_eval_df, verbose = 0)
        if y_pred_proba.ndim == 2 and y_pred_proba.shape[1] == 1:
            y_pred = (y_pred_proba.ravel() >= 0.5).astype(int)
            y_proba = y_pred_proba.ravel()
        else:
            y_pred = np.argmax(y_pred_proba, axis=1)
            y_proba = y_pred_proba
        return y_pred, y_proba

    if framework == 'xgboost':
        try:
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_eval_df)
                if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                    y_pred = np.argmax(y_proba, axis = 1)
                    y_proba_for_auc = y_proba
                else:
                    y_pred = (y_proba.ravel() > 0.5).astype(int)
                    y_proba_for_auc = y_proba.ravel()
                return y_pred, y_proba_for_auc

            if isinstance(model, xgb.core.Booster):
                dmat = xgb.DMatrix(X_eval_df)
                probs = model.predict(dmat)
                if probs.ndim == 1:
                    y_pred = (probs > 0.5).astype(int)
                    return y_pred, probs.ravel()
                else:
                    y_pred = np.argmax(probs, axis=1)
                    return y_pred, probs
        except Exception:
            pass

        y_pred_raw = model.predict(X_eval_df)
        if isinstance(y_pred_raw[0], str):
            y_pred = label_encoder.transform(y_pred_raw)
        else:
            y_pred = np.asarray(y_pred_raw).astype(int)
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                return y_pred, y_proba
            else:
                return y_pred, y_proba.ravel()
        except Exception:
            return y_pred, None

    y_pred_raw = model.predict(X_eval_df)
    if isinstance(y_pred_raw[0], str):
        y_pred = label_encoder.transform(y_pred_raw)
    else:
        y_pred = np.asarray(y_pred_raw).astype(int)

    y_proba_for_auc = None
    if hasattr(model, 'predict_proba'):
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                y_proba_for_auc = y_proba
            else:
                y_proba_for_auc = y_proba.ravel()
        except Exception:
            y_proba_for_auc = None
    elif hasattr(model, 'decision_function'):
        try:
            dec = model.decision_function(X_eval_df)
            dec = np.ravel(dec)
            if dec.max() != dec.min():
                y_proba_for_auc = (dec - dec.min()) / (dec.max() - dec.min())
            else:
                y_proba_for_auc = dec
        except Exception:
            y_proba_for_auc = None

    return y_pred, y_proba_for_auc

# Main loop
f1_average = 'macro'
predictions_dict = {}

for model_name, path, framework, data_type, used_smote in models_to_log:
    print('\n' + '=' * 70)
    print(f'Loading {model_name}...')

    model = None
    xgb_params_loaded = None
    try:
        if framework == 'keras':
            model = keras.models.load_model(path)
        elif framework == 'xgboost':
            try:
                model = joblib.load(path)
            except Exception:
                try:
                    clf = xgb.XGBClassifier()
                    clf.load_model(path)
                    model = clf
                except Exception:
                    try:
                        booster = xgb.Booster()
                        booster.load_model(path)
                        model = booster
                    except Exception as e:
                        raise e
            params_path = path.replace('.json', '_params.json')
            if os.path.exists(params_path):
                try:
                    with open(params_path, 'r') as f:
                        xgb_params_loaded = json.load(f)
                except Exception:
                    xgb_params_loaded = None
        else:
            model = joblib.load(path)
    except Exception as e:
        print(f'Error loading {model_name}: {e}')
        continue

    if model_name == 'Stacking Classifier':
        X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
        reason = "Stacking Classifier: Using original 'X_test' (trained that way)"
    else:
        if framework != 'keras' and model_includes_scaler(model):
            X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
            reason = "Pipeline includes scaler: Using original 'X_test' (pipeline will scale)"
        else:
            X_eval = X_test_scaled if data_type == 'scaled' else X_test
            X_eval_df = ensure_dataframe(X_eval, ref_columns = X_train.columns)
            reason = f"Data type = '{data_type}': Using '{'X_test_scaled' if data_type == 'scaled' else 'X_test'}'"

    print('  ->', reason)

    try:
        y_pred, y_proba_for_auc = get_predictions_and_probas(model, framework, X_eval_df, label_encoder)
    except Exception as e:
        print(f'Prediction failed for {model_name}: {e}')
        continue

    y_true_enc = Y_test_encoded
    acc = accuracy_score(y_true_enc, y_pred)
    f1 = f1_score(y_true_enc, y_pred, average = f1_average)

    auc_val = None
    try:
        n_classes = len(np.unique(y_true_enc))
        if n_classes == 2 and y_proba_for_auc is not None:
            if y_proba_for_auc.ndim == 2 and y_proba_for_auc.shape[1] == 2:
                auc_val = roc_auc_score(y_true_enc, y_proba_for_auc[:, 1])
            else:
                auc_val = roc_auc_score(y_true_enc, y_proba_for_auc)
        else:
            # No ROC AUC for multiclass
            auc_val = None
    except Exception as e:
        print(f"  -> Warning: roc_auc_score failed: {e}; y_proba shape: {np.shape(y_proba_for_auc) if y_proba_for_auc is not None else 'None'}")
        auc_val = None

    print(f"  -> Accuracy: {acc:.4f}, F1 ({f1_average}): {f1:.4f}, ROC AUC: {auc_val if auc_val is not None else 'None'}")
    print('  -> Sample preds (first 10):', list(y_pred[:10]))
    unique, counts = np.unique(y_pred, return_counts = True)
    print('  -> Unique pred counts:', dict(zip(unique.tolist(), counts.tolist())))

    predictions_dict[model_name] = y_pred

    run_name = f'{model_name} Multiclass (Charging State)'
    with mlflow.start_run(run_name = run_name) as run:
        row = df_prod[df_prod['Model'] == model_name]
        if not row.empty:
            mlflow.log_metric('Prediction Time secs', float(row['Prediction Time (sec)'].iloc[0]))
          # mlflow.log_metric('Memory Used (kB)', float(row['Memory Used (kB)'].iloc[0]))
            mlflow.log_metric('CPU Time secs', float(row['CPU Time (sec)'].iloc[0]))
            mlflow.log_metric('Physical Cores', int(row['Physical Cores'].iloc[0]))
            mlflow.log_metric('Logical Cores', int(row['Logical Cores'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Raw', float(row['CPU % Used (Raw)'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Normalized', float(row['CPU % Used (Normalized)'].iloc[0]))
            mlflow.log_metric('Model Size MB', float(row['Model Size (MB)'].iloc[0]))
          # mlflow.log_metric('F1 Macro (CSV)', float(row['F1 Macro'])) # I log it from the model not the csv

        mlflow.log_metric('Accuracy', float(f'{acc:.4f}'))
        mlflow.log_metric('F1 macro', float(f'{f1:.4f}'))
        if auc_val is not None:
            mlflow.log_metric('ROC AUC', float(f'{auc_val:.4f}'))

        mlflow.log_param('SMOTE', bool(used_smote))
        mlflow.set_tag('SMOTE', str(bool(used_smote)))
        mlflow.log_param('Framework', framework)
        mlflow.log_param('ModelPath', path)

        try:
            if framework == 'keras':
                kp = keras_params_summary(model)
                safe_kp = _safe_params_dict(kp, prefix = 'keras_')
                if safe_kp:
                    mlflow.log_params(safe_kp)

            elif framework == 'xgboost':
                if isinstance(model, Pipeline):
                    inner = extract_xgb_from_pipeline(model)
                    if inner:
                        try:
                            safe = safe_log_params_iter(inner, prefix = 'xgb_')
                            if safe:
                                mlflow.log_params(safe)
                        except Exception:
                            pass
                else:
                    try:
                        if hasattr(model, 'get_xgb_params'):
                            xp = model.get_xgb_params()
                            mlflow.log_params(_safe_params_dict(xp, prefix = 'xgb_'))
                    except Exception:
                        pass
                    try:
                        if hasattr(model, 'get_params'):
                            gp = model.get_params()
                            safe_gp = _safe_params_dict(gp, prefix = 'xgb_')
                            if safe_gp:
                                mlflow.log_params(safe_gp)
                    except Exception:
                        pass
                    if xgb_params_loaded:
                        mlflow.log_params(_safe_params_dict(xgb_params_loaded, prefix = 'xgb_file_'))

            elif 'stack' in model_name.lower() or hasattr(model, 'final_estimator_'):
                try:
                    if hasattr(model, 'final_estimator_'):
                        try:
                            meta_params = safe_log_params_iter(model.final_estimator_, prefix = 'meta_')
                            if meta_params:
                                mlflow.log_params(meta_params)
                        except Exception:
                            pass
                    if hasattr(model, 'named_estimators_'):
                        for name, est in model.named_estimators_.items():
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{name}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                pass
                    elif hasattr(model, 'estimators_'):
                        for i, est in enumerate(model.estimators_):
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{i}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                pass
                except Exception as e:
                    print('  -> Warning logging stacking params:', e)

            else:
                if hasattr(model, 'get_params'):
                    try:
                        gp = model.get_params()
                        mlflow.log_params(_safe_params_dict(gp))
                    except Exception:
                        if isinstance(model, Pipeline):
                            for name, step in model.named_steps.items():
                                if hasattr(step, 'get_params'):
                                    mlflow.log_params(_safe_params_dict(step.get_params(), prefix = f'{name}__'))
        except Exception as e:
            print('  -> Warning while preparing params to log:', e)

        try:
            target_names = label_encoder.classes_ if 'label_encoder' in globals() else None
            creport_dict = classification_report(y_true_enc, y_pred, output_dict = True)
            creport_text = classification_report(y_true_enc, y_pred, target_names = target_names)
            mlflow.log_text(json.dumps(creport_dict, indent = 4), 'classification_report.json')
            mlflow.log_text(creport_text, 'classification_report.txt')
        except Exception as e:
            print('  -> Warning logging classification report:', e)

        try:
            logged_ok = mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = X_eval_df)
            if not logged_ok:
                print('  -> Warning: model artifact not logged via standard mlflow path and fallback.')
        except Exception as e:
            print('  -> Error logging model artifact:', e)

        try:
            run_id = run.info.run_id
            exp_id = run.info.experiment_id
            print(f'  -> MLflow run id: {run_id} (experiment {exp_id})')
        except Exception:
            pass

    print(f'✅ Logged {model_name} (SMOTE = {used_smote})')


# After loop: cross-model identical prediction check
print('\n' + '=' * 70)
print('Cross-model prediction overlap:')
names = list(predictions_dict.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = names[i], names[j]
        overlap = np.mean(predictions_dict[a] == predictions_dict[b])
        print(f'  - {a} vs {b}: identical fraction = {overlap:.3f}')


Loading KNN...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.8260, F1 (macro): 0.8077, ROC AUC: None
  -> Sample preds (first 10): [0, 0, 1, 0, 0, 0, 0, 0, 2, 2]
  -> Unique pred counts: {0: 3037, 1: 1112, 2: 1770}
  -> MLflow run id: 9441b85cc72f4a26a21fb246c3910faf (experiment 1)
🏃 View run KNN Multiclass (Charging State) at: http://127.0.0.1:5000/#/experiments/1/runs/9441b85cc72f4a26a21fb246c3910faf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
✅ Logged KNN (SMOTE = False)

Loading KNN SMOTE...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.8081, F1 (macro): 0.8049, ROC AUC: None
  -> Sample preds (first 10): [0, 0, 1, 0, 0, 1, 0, 0, 2, 2]
  -> Unique pred counts: {0: 2517, 1: 1622, 2: 1780}
  -> MLflow run id: 86f09a3895aa4207ab3b787300511df6 (experiment 1)
🏃 View run KNN SMOTE Multiclass (Charging State) at: http://127.0.0.1:5000/#/experiments/1/runs/86f09a3895aa4207ab3b78

# Multiclass Classification (Both States)

In [16]:
# Set the experiment in MLflow
mlflow.set_experiment('Multiclass Classification (Both States)')

<Experiment: artifact_location='/Users/yannis/mlflow-artifacts/2', creation_time=1761295858748, experiment_id='2', last_update_time=1761297936105, lifecycle_stage='active', name='Multiclass Classification (Both States)', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [17]:
# Load the sheet into a DataFrame
df = pd.read_csv(DATA_PATH)

# Display basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115298 entries, 0 to 115297
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   time           115298 non-null  object 
 1   shunt_voltage  115298 non-null  int64  
 2   bus_voltage_V  115298 non-null  float64
 3   current_mA     115298 non-null  int64  
 4   power_mW       115298 non-null  int64  
 5   State          115298 non-null  object 
 6   Attack         115298 non-null  object 
 7   Attack-Group   115298 non-null  object 
 8   Label          115298 non-null  object 
 9   interface      115298 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 8.8+ MB


In [18]:
# Print the first rows of the DataFrame
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp


In [19]:
# Define mapping dictionary
attack_mapping = {
    'syn-flood': 'DoS',
    'tcp-flood': 'DoS',
    'cryptojacking': 'Cryptojacking',
    'vuln-scan': 'Recon',
    'syn-stealth': 'Recon',
    'Backdoor': 'Backdoor',
    'none': 'Benign'
}

# Apply mapping to create 'Attack-New' column
df['Attack-New'] = df['Attack'].map(attack_mapping)

# Print the first rows of the DataFrame to verify the new column
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp,DoS
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp,DoS
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp,DoS
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp,DoS
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp,DoS


In [20]:
# Select numerical features for classification (excluding categorical columns)
numerical_features = ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']

# Define features (X) and target variable (Y)
X = df[numerical_features]
Y = df['Attack-New']

In [21]:
# Split the dataset into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42, stratify = Y, shuffle = True)

In [22]:
# Split the training set into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.2, random_state = 42, stratify = Y_train, shuffle = True)

In [23]:
# Set the encoder
label_encoder = LabelEncoder()

# Encode 'Attack-New' labels as numbers for each subset
# Note: This is done after splitting to avoid data leakage
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_val_encoded = label_encoder.transform(Y_val)
Y_test_encoded = label_encoder.transform(Y_test)

# Check what each number represents
print('Label encoder classes:', label_encoder.classes_)

Label encoder classes: ['Backdoor' 'Benign' 'Cryptojacking' 'DoS' 'Recon']


In [24]:
# Create the scaler
scaler = StandardScaler()

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Shape of the scaled data
print(f'Shape of X_train_scaled: {X_train_scaled.shape}')
print(f'Shape of X_val_scaled: {X_val_scaled.shape}')
print(f'Shape of X_test_scaled: {X_test_scaled.shape}')

Shape of X_train_scaled: (73790, 4)
Shape of X_val_scaled: (18448, 4)
Shape of X_test_scaled: (23060, 4)


In [ ]:
# Suppress specific warnings for cleaner output
warnings.filterwarnings('ignore', category = UserWarning)

# Models list
models_to_log = [
    ('KNN', MODELS_DIR + 'knn_multi_both_best_model.pkl', 'sklearn', 'scaled', False),
    ('KNN SMOTE', MODELS_DIR + 'knn_multi_both_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('XGBoost', MODELS_DIR + 'xgboost_multi_both_best_model.json', 'xgboost', 'scaled', False),
    ('XGBoost SMOTE', MODELS_DIR + 'xgboost_multi_both_smote_best_model.json', 'xgboost', 'scaled', True),
    ('Logistic Regression', MODELS_DIR + 'logistic_regression_multi_both_best_model.pkl', 'sklearn', 'scaled', False),
    ('Logistic Regression SMOTE', MODELS_DIR + 'logistic_regression_multi_both_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('Random Forest', MODELS_DIR + 'random_forest_multi_both_best_model.pkl', 'sklearn', 'original', False),
    ('Random Forest SMOTE', MODELS_DIR + 'random_forest_multi_both_smote_best_model.pkl', 'sklearn', 'original', True),
    ('ANN', MODELS_DIR + 'ann_multi_both_model.keras', 'keras', 'scaled', False),
    ('ANN SMOTE', MODELS_DIR + 'ann_multi_both_smote_model.keras', 'keras', 'scaled', True),
    ('Stacking Classifier', MODELS_DIR + 'stacking_clf_multi_both_best_model.pkl', 'sklearn', 'original', False),
]

# Load production results CSV
# Clean CSV columns for MLflow logging
df_prod = pd.read_csv('df_production_results_multi_both.csv')

# Ensure numeric
for col in ['Prediction Time (sec)','F1 Macro', 'Memory Used (kB)', 'CPU Time (sec)', 'Physical Cores', 'Logical Cores', 'CPU % Used (Raw)',
            'CPU % Used (Normalized)','Model Size (MB)']:
    df_prod[col] = pd.to_numeric(df_prod[col], errors = 'coerce')

# Sanity checks
required_globals = ['X_train', 'X_test', 'X_test_scaled', 'Y_test_encoded', 'label_encoder']
missing = [g for g in required_globals if g not in globals()]
if missing:
    raise RuntimeError(f'Please ensure these globals exist before running: {missing}')

# Helper functions
def ensure_dataframe(X, ref_columns):
    if isinstance(X, np.ndarray):
        return pd.DataFrame(X, columns = list(ref_columns))
    if isinstance(X, pd.DataFrame):
        if list(X.columns) != list(ref_columns) and X.shape[1] == len(ref_columns):
            return pd.DataFrame(X.values, columns = list(ref_columns))
        return X
    raise ValueError('X must be numpy array or pandas DataFrame')

def model_includes_scaler(model):
    if isinstance(model, Pipeline):
        for step in model.named_steps.values():
            cls_name = step.__class__.__name__.lower()
            if 'scaler' in cls_name or 'standardscaler' in cls_name or 'minmax' in cls_name or 'preprocessor' in cls_name:
                return True
    return False

def _safe_params_dict(params: dict, prefix = ''):
    safe = {}
    for k, v in params.items():
        key = f'{prefix}{k}'
        if isinstance(v, (str, int, float, bool)) or v is None:
            safe[key] = v
        else:
            try:
                s = json.dumps(v) if isinstance(v, (dict, list)) else str(v)
                safe[key] = s[:200]
            except Exception:
                safe[key] = str(type(v))
    return safe

def keras_params_summary(model: keras.Model):
    params = {}
    try:
        params['keras_total_params'] = model.count_params()
    except Exception:
        params['keras_total_params'] = None
    params['keras_layers_count'] = len(model.layers)
    for i, layer in enumerate(model.layers):
        try:
            cfg = layer.get_config()
        except Exception:
            cfg = {}
        params[f'layer_{i}_class'] = layer.__class__.__name__
        if 'units' in cfg:
            params[f'layer_{i}_units'] = cfg.get('units')
        if 'activation' in cfg:
            params[f'layer_{i}_activation'] = cfg.get('activation')
    try:
        opt = model.optimizer.get_config()
        params['optimizer'] = model.optimizer.__class__.__name__
        for k, v in opt.items():
            params[f'opt_{k}'] = v if isinstance(v, (str, int, float, bool)) else str(v)
    except Exception:
        pass
    try:
        params['loss'] = model.loss if not callable(model.loss) else str(model.loss)
    except Exception:
        pass
    return params

def extract_xgb_from_pipeline(m):
    if not isinstance(m, Pipeline):
        return None
    for step in m.named_steps.values():
        cls = step.__class__.__name__.lower()
        if 'xgb' in cls or hasattr(step, 'get_xgb_params'):
            return step
    return None

def safe_head_example(X, n=8):
    try:
        if isinstance(X, pd.DataFrame):
            return X.head(n)
        if isinstance(X, np.ndarray):
            return pd.DataFrame(X[:n], columns = X_train.columns)
    except Exception:
        pass
    return None

def safe_predict_for_signature(model, X_example, framework):
    try:
        fw = (framework or '').lower()
        if fw == 'keras':
            preds = model.predict(X_example, verbose = 0)
            return preds
        elif fw == 'xgboost':
            if isinstance(model, xgb.core.Booster):
                d = xgb.DMatrix(X_example)
                preds = model.predict(d)
                return preds
            else:
                if hasattr(model, 'predict'):
                    return model.predict(X_example)
        else:
            return model.predict(X_example)
    except Exception:
        return None

def safe_log_params_iter(obj, prefix = ''):
    safe = {}
    try:
        gp = obj.get_params()
        for k, v in gp.items():
            try:
                if isinstance(v, (list, dict)):
                    safe[f'{prefix}{k}'] = json.dumps(v)[:400]
                elif hasattr(v, '__name__') and getattr(v, '__name__'):
                    safe[f'{prefix}{k}'] = getattr(v, '__name__')
                else:
                    safe[f'{prefix}{k}'] = v
            except Exception:
                safe[f'{prefix}{k}'] = str(type(v))
    except Exception:
        pass
    return safe

def mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = None):
    fw = (framework or '').lower()
    example_df = safe_head_example(X_example) if X_example is not None else None

    input_example_np = None
    signature = None
    input_example = None

    if example_df is not None:
        input_example_np = example_df.values if isinstance(example_df, pd.DataFrame) else example_df
        input_example = input_example_np
        preds = safe_predict_for_signature(model, input_example_np, fw)
        try:
            signature = infer_signature(input_example_np, preds) if preds is not None else infer_signature(input_example_np, None)
        except Exception:
            signature = None

    with tempfile.TemporaryDirectory(prefix = 'mlflow_model_') as tmp_path:
        try:
            if fw == 'keras':
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                # Skip passing input_example here to avoid MLflow directory error
                try:
                    mlflow.keras.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception as e:
                    print('  -> Primary keras log_model failed:', e)
                    saved_path = os.path.join(tmp_path, 'model.keras')
                    os.makedirs(saved_path, exist_ok = True)
                    try:
                        model.save(saved_path, include_optimizer = True)
                    except Exception:
                        model.save(saved_path)
                    mlflow.log_artifact(saved_path, artifact_path = artifact_name)
                    return True

            elif fw == 'xgboost':
                inner = extract_xgb_from_pipeline(model) if isinstance(model, Pipeline) else None
                candidate = inner or model
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.xgboost.log_model(xgb_model = candidate, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                try:
                    if isinstance(candidate, xgb.core.Booster):
                        p = os.path.join(tmp_path, 'booster.json')
                        candidate.save_model(p)
                        mlflow.log_artifact(p, artifact_path = artifact_name)
                        return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

            else:
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.sklearn.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

        except Exception as e:
            print(f'mlflow_log_model_auto: primary path exception: {e}')
            return False

# Unified prediction/proba extractor (works for pipelines, sklearn, keras, xgboost)
def get_predictions_and_probas(model, framework, X_eval_df, label_encoder):
    if framework == 'keras':
        y_pred_proba = model.predict(X_eval_df, verbose = 0)
        if y_pred_proba.ndim == 2 and y_pred_proba.shape[1] == 1:
            y_pred = (y_pred_proba.ravel() >= 0.5).astype(int)
            y_proba = y_pred_proba.ravel()
        else:
            y_pred = np.argmax(y_pred_proba, axis=1)
            y_proba = y_pred_proba
        return y_pred, y_proba

    if framework == 'xgboost':
        try:
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_eval_df)
                if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                    y_pred = np.argmax(y_proba, axis=1)
                    y_proba_for_auc = y_proba
                else:
                    y_pred = (y_proba.ravel() > 0.5).astype(int)
                    y_proba_for_auc = y_proba.ravel()
                return y_pred, y_proba_for_auc

            if isinstance(model, xgb.core.Booster):
                dmat = xgb.DMatrix(X_eval_df)
                probs = model.predict(dmat)
                if probs.ndim == 1:
                    y_pred = (probs > 0.5).astype(int)
                    return y_pred, probs.ravel()
                else:
                    y_pred = np.argmax(probs, axis = 1)
                    return y_pred, probs
        except Exception:
            pass

        y_pred_raw = model.predict(X_eval_df)
        if isinstance(y_pred_raw[0], str):
            y_pred = label_encoder.transform(y_pred_raw)
        else:
            y_pred = np.asarray(y_pred_raw).astype(int)
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                return y_pred, y_proba
            else:
                return y_pred, y_proba.ravel()
        except Exception:
            return y_pred, None

    y_pred_raw = model.predict(X_eval_df)
    if isinstance(y_pred_raw[0], str):
        y_pred = label_encoder.transform(y_pred_raw)
    else:
        y_pred = np.asarray(y_pred_raw).astype(int)

    y_proba_for_auc = None
    if hasattr(model, 'predict_proba'):
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                y_proba_for_auc = y_proba
            else:
                y_proba_for_auc = y_proba.ravel()
        except Exception:
            y_proba_for_auc = None
    elif hasattr(model, 'decision_function'):
        try:
            dec = model.decision_function(X_eval_df)
            dec = np.ravel(dec)
            if dec.max() != dec.min():
                y_proba_for_auc = (dec - dec.min()) / (dec.max() - dec.min())
            else:
                y_proba_for_auc = dec
        except Exception:
            y_proba_for_auc = None

    return y_pred, y_proba_for_auc

# Main loop
f1_average = 'macro'
predictions_dict = {}

for model_name, path, framework, data_type, used_smote in models_to_log:
    print('\n' + '=' * 70)
    print(f'Loading {model_name}...')

    model = None
    xgb_params_loaded = None
    try:
        if framework == 'keras':
            model = keras.models.load_model(path)
        elif framework == 'xgboost':
            try:
                model = joblib.load(path)
            except Exception:
                try:
                    clf = xgb.XGBClassifier()
                    clf.load_model(path)
                    model = clf
                except Exception:
                    try:
                        booster = xgb.Booster()
                        booster.load_model(path)
                        model = booster
                    except Exception as e:
                        raise e
            params_path = path.replace('.json', '_params.json')
            if os.path.exists(params_path):
                try:
                    with open(params_path, 'r') as f:
                        xgb_params_loaded = json.load(f)
                except Exception:
                    xgb_params_loaded = None
        else:
            model = joblib.load(path)
    except Exception as e:
        print(f'Error loading {model_name}: {e}')
        continue

    if model_name == 'Stacking Classifier':
        X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
        reason = "Stacking Classifier: Using original 'X_test' (trained that way)"
    else:
        if framework != 'keras' and model_includes_scaler(model):
            X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
            reason = "Pipeline includes scaler: Using original 'X_test' (pipeline will scale)"
        else:
            X_eval = X_test_scaled if data_type == 'scaled' else X_test
            X_eval_df = ensure_dataframe(X_eval, ref_columns = X_train.columns)
            reason = f"Data type = '{data_type}': Using '{'X_test_scaled' if data_type == 'scaled' else 'X_test'}'"

    print('  ->', reason)

    try:
        y_pred, y_proba_for_auc = get_predictions_and_probas(model, framework, X_eval_df, label_encoder)
    except Exception as e:
        print(f'Prediction failed for {model_name}: {e}')
        continue

    y_true_enc = Y_test_encoded
    acc = accuracy_score(y_true_enc, y_pred)
    f1 = f1_score(y_true_enc, y_pred, average = f1_average)

    auc_val = None
    try:
        n_classes = len(np.unique(y_true_enc))
        if n_classes == 2 and y_proba_for_auc is not None:
            if y_proba_for_auc.ndim == 2 and y_proba_for_auc.shape[1] == 2:
                auc_val = roc_auc_score(y_true_enc, y_proba_for_auc[:, 1])
            else:
                auc_val = roc_auc_score(y_true_enc, y_proba_for_auc)
        else:
            # No ROC AUC for multiclass
            auc_val = None
    except Exception as e:
        print(f"  -> Warning: roc_auc_score failed: {e}; y_proba shape: {np.shape(y_proba_for_auc) if y_proba_for_auc is not None else 'None'}")
        auc_val = None

    print(f"  -> Accuracy: {acc:.4f}, F1 ({f1_average}): {f1:.4f}, ROC AUC: {auc_val if auc_val is not None else 'None'}")
    print('  -> Sample preds (first 10):', list(y_pred[:10]))
    unique, counts = np.unique(y_pred, return_counts = True)
    print('  -> Unique pred counts:', dict(zip(unique.tolist(), counts.tolist())))

    predictions_dict[model_name] = y_pred

    run_name = f'{model_name} Multiclass (Both States)'
    with mlflow.start_run(run_name = run_name) as run:
        row = df_prod[df_prod['Model'] == model_name]
        if not row.empty:
            mlflow.log_metric('Prediction Time secs', float(row['Prediction Time (sec)'].iloc[0]))
          # mlflow.log_metric('Memory Used (kB)', float(row['Memory Used (kB)'].iloc[0]))
            mlflow.log_metric('CPU Time secs', float(row['CPU Time (sec)'].iloc[0]))
            mlflow.log_metric('Physical Cores', int(row['Physical Cores'].iloc[0]))
            mlflow.log_metric('Logical Cores', int(row['Logical Cores'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Raw', float(row['CPU % Used (Raw)'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Normalized', float(row['CPU % Used (Normalized)'].iloc[0]))
            mlflow.log_metric('Model Size MB', float(row['Model Size (MB)'].iloc[0]))
          # mlflow.log_metric('F1 Macro (CSV)', float(row['F1 Macro'])) # I log it from the model not the csv
        
        mlflow.log_metric('Accuracy', float(f'{acc:.4f}'))
        mlflow.log_metric('F1 macro', float(f'{f1:.4f}'))
        if auc_val is not None:
            mlflow.log_metric('ROC AUC', float(f'{auc_val:.4f}'))

        mlflow.log_param('SMOTE', bool(used_smote))
        mlflow.set_tag('SMOTE', str(bool(used_smote)))
        mlflow.log_param('Framework', framework)
        mlflow.log_param('ModelPath', path)

        try:
            if framework == 'keras':
                kp = keras_params_summary(model)
                safe_kp = _safe_params_dict(kp, prefix = 'keras_')
                if safe_kp:
                    mlflow.log_params(safe_kp)

            elif framework == 'xgboost':
                if isinstance(model, Pipeline):
                    inner = extract_xgb_from_pipeline(model)
                    if inner:
                        try:
                            safe = safe_log_params_iter(inner, prefix = 'xgb_')
                            if safe:
                                mlflow.log_params(safe)
                        except Exception:
                            pass
                else:
                    try:
                        if hasattr(model, 'get_xgb_params'):
                            xp = model.get_xgb_params()
                            mlflow.log_params(_safe_params_dict(xp, prefix = 'xgb_'))
                    except Exception:
                        pass
                    try:
                        if hasattr(model, 'get_params'):
                            gp = model.get_params()
                            safe_gp = _safe_params_dict(gp, prefix = 'xgb_')
                            if safe_gp:
                                mlflow.log_params(safe_gp)
                    except Exception:
                        pass
                    if xgb_params_loaded:
                        mlflow.log_params(_safe_params_dict(xgb_params_loaded, prefix = 'xgb_file_'))

            elif 'stack' in model_name.lower() or hasattr(model, 'final_estimator_'):
                try:
                    if hasattr(model, 'final_estimator_'):
                        try:
                            meta_params = safe_log_params_iter(model.final_estimator_, prefix = 'meta_')
                            if meta_params:
                                mlflow.log_params(meta_params)
                        except Exception:
                            pass
                    if hasattr(model, 'named_estimators_'):
                        for name, est in model.named_estimators_.items():
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{name}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                pass
                    elif hasattr(model, 'estimators_'):
                        for i, est in enumerate(model.estimators_):
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{i}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                pass
                except Exception as e:
                    print('  -> Warning logging stacking params:', e)

            else:
                if hasattr(model, 'get_params'):
                    try:
                        gp = model.get_params()
                        mlflow.log_params(_safe_params_dict(gp))
                    except Exception:
                        if isinstance(model, Pipeline):
                            for name, step in model.named_steps.items():
                                if hasattr(step, 'get_params'):
                                    mlflow.log_params(_safe_params_dict(step.get_params(), prefix = f'{name}__'))
        except Exception as e:
            print('  -> Warning while preparing params to log:', e)

        try:
            target_names = label_encoder.classes_ if 'label_encoder' in globals() else None
            creport_dict = classification_report(y_true_enc, y_pred, output_dict = True)
            creport_text = classification_report(y_true_enc, y_pred, target_names = target_names)
            mlflow.log_text(json.dumps(creport_dict, indent = 4), 'classification_report.json')
            mlflow.log_text(creport_text, 'classification_report.txt')
        except Exception as e:
            print('  -> Warning logging classification report:', e)

        try:
            logged_ok = mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = X_eval_df)
            if not logged_ok:
                print('  -> Warning: model artifact not logged via standard mlflow path and fallback.')
        except Exception as e:
            print('  -> Error logging model artifact:', e)

        try:
            run_id = run.info.run_id
            exp_id = run.info.experiment_id
            print(f'  -> MLflow run id: {run_id} (experiment {exp_id})')
        except Exception:
            pass

    print(f'✅ Logged {model_name} (SMOTE = {used_smote})')


# After loop: cross-model identical prediction check
print('\n' + '=' * 70)
print('Cross-model prediction overlap:')
names = list(predictions_dict.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = names[i], names[j]
        overlap = np.mean(predictions_dict[a] == predictions_dict[b])
        print(f'  - {a} vs {b}: identical fraction = {overlap:.3f}')


Loading KNN...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.6110, F1 (macro): 0.6374, ROC AUC: None
  -> Sample preds (first 10): [3, 4, 1, 4, 1, 2, 3, 1, 1, 4]
  -> Unique pred counts: {0: 5099, 1: 2660, 2: 2426, 3: 5315, 4: 7560}
  -> MLflow run id: 5501649acaf74ac7adfccded721c3917 (experiment 2)
🏃 View run KNN Multiclass (Both States) at: http://127.0.0.1:5000/#/experiments/2/runs/5501649acaf74ac7adfccded721c3917
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
✅ Logged KNN (SMOTE = False)

Loading KNN SMOTE...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.5933, F1 (macro): 0.6170, ROC AUC: None
  -> Sample preds (first 10): [3, 4, 1, 4, 1, 2, 2, 1, 1, 4]
  -> Unique pred counts: {0: 4912, 1: 3229, 2: 3305, 3: 4395, 4: 7219}
  -> MLflow run id: 76e0f6de5e3d48bea16f340d722473cf (experiment 2)
🏃 View run KNN SMOTE Multiclass (Both States) at: http://127.0.0.1:5000/#/experiments

# Binary Classification (Charging State)

In [26]:
# Set the experiment in MLflow
mlflow.set_experiment('Binary Classification (Charging State)')

<Experiment: artifact_location='/Users/yannis/mlflow-artifacts/3', creation_time=1761300037818, experiment_id='3', last_update_time=1761300037818, lifecycle_stage='active', name='Binary Classification (Charging State)', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [27]:
# Load the sheet into a DataFrame
df = pd.read_csv(DATA_PATH)

# Display basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115298 entries, 0 to 115297
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   time           115298 non-null  object 
 1   shunt_voltage  115298 non-null  int64  
 2   bus_voltage_V  115298 non-null  float64
 3   current_mA     115298 non-null  int64  
 4   power_mW       115298 non-null  int64  
 5   State          115298 non-null  object 
 6   Attack         115298 non-null  object 
 7   Attack-Group   115298 non-null  object 
 8   Label          115298 non-null  object 
 9   interface      115298 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 8.8+ MB


In [28]:
# Define mapping dictionary
attack_mapping = {
    'syn-flood': 'DoS',
    'tcp-flood': 'DoS',
    'cryptojacking': 'Cryptojacking',
    'vuln-scan': 'Recon',
    'syn-stealth': 'Recon',
    'Backdoor': 'Backdoor',
    'none': 'Benign'
}

# Apply mapping to create 'Attack-New' column
df['Attack-New'] = df['Attack'].map(attack_mapping)

# Print the first rows of the DataFrame to verify the new column
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp,DoS
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp,DoS
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp,DoS
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp,DoS
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp,DoS


In [29]:
# Filter data for 'charging' state
df_charging = df[df['State'] == 'charging'].copy()

# Print the first few rows of the filtered DataFrame
df_charging.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New
34884,12/26/2023 15:17,660,5.185,652,3380,charging,none,none,benign,none,Benign
34885,12/26/2023 15:17,769,5.189,675,3500,charging,none,none,benign,none,Benign
34886,12/26/2023 15:17,525,5.197,512,2660,charging,none,none,benign,none,Benign
34887,12/26/2023 15:17,690,5.181,642,3340,charging,none,none,benign,none,Benign
34888,12/26/2023 15:17,638,5.189,574,2780,charging,none,none,benign,none,Benign


In [30]:
# Convert "Attack-New" to binary: 1 = Attack, 0 = Benign
df_charging['Attack-Binary'] = df_charging['Attack-New'].apply(lambda x: 0 if x == 'Benign' else 1)

# Print the first rows of the DataFrame to verify the new column
df_charging.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New,Attack-Binary
34884,12/26/2023 15:17,660,5.185,652,3380,charging,none,none,benign,none,Benign,0
34885,12/26/2023 15:17,769,5.189,675,3500,charging,none,none,benign,none,Benign,0
34886,12/26/2023 15:17,525,5.197,512,2660,charging,none,none,benign,none,Benign,0
34887,12/26/2023 15:17,690,5.181,642,3340,charging,none,none,benign,none,Benign,0
34888,12/26/2023 15:17,638,5.189,574,2780,charging,none,none,benign,none,Benign,0


In [31]:
# Select numerical features for classification (excluding categorical columns)
numerical_features = ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']

# Define features (X) and target variable (Y)
X = df_charging[numerical_features]
Y = df_charging['Attack-Binary']

In [32]:
# Split the dataset into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42, stratify = Y, shuffle = True)

In [33]:
# Split the training set into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.2, random_state = 42, stratify = Y_train, shuffle = True)

In [34]:
# Set the encoder 
label_encoder = LabelEncoder()

# Encode 'Attack-New' labels as numbers for each subset
# Note: This is done after splitting to avoid data leakage
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_val_encoded = label_encoder.transform(Y_val)
Y_test_encoded = label_encoder.transform(Y_test)

# Check what each number represents
print('Label encoder classes:', label_encoder.classes_)

Label encoder classes: [0 1]


In [35]:
# Create the scaler
scaler = StandardScaler()

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Shape of the scaled data
print(f'Shape of X_train_scaled: {X_train_scaled.shape}')
print(f'Shape of X_val_scaled: {X_val_scaled.shape}')
print(f'Shape of X_test_scaled: {X_test_scaled.shape}')

Shape of X_train_scaled: (18939, 4)
Shape of X_val_scaled: (4735, 4)
Shape of X_test_scaled: (5919, 4)


In [ ]:
# Suppress specific warnings for cleaner output
warnings.filterwarnings('ignore', category = UserWarning)

# Models list
models_to_log = [
    ('KNN', MODELS_DIR + 'knn_bin_charging_best_model.pkl', 'sklearn', 'scaled', False),
    ('KNN SMOTE', MODELS_DIR + 'knn_bin_charging_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('RBF SVM', MODELS_DIR + 'rbf_svm_bin_charging_best_model.pkl', 'sklearn', 'scaled', False),
    ('RBF SVM SMOTE', MODELS_DIR + 'rbf_svm_bin_charging_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('XGBoost', MODELS_DIR + 'xgboost_bin_charging_best_model.json', 'xgboost', 'scaled', False),
    ('XGBoost SMOTE', MODELS_DIR + 'xgboost_bin_charging_smote_best_model.json', 'xgboost', 'scaled', True),
    ('Logistic Regression', MODELS_DIR + 'logistic_regression_bin_charging_best_model.pkl', 'sklearn', 'scaled', False),
    ('Logistic Regression SMOTE', MODELS_DIR + 'logistic_regression_bin_charging_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('Random Forest', MODELS_DIR + 'random_forest_bin_charging_best_model.pkl', 'sklearn', 'original', False),
    ('Random Forest SMOTE', MODELS_DIR + 'random_forest_bin_charging_smote_best_model.pkl', 'sklearn', 'original', True),
    ('ANN', MODELS_DIR + 'ann_bin_charging_model.keras', 'keras', 'scaled', False),
    ('ANN Weighted', MODELS_DIR + 'ann_bin_charging_weighted_model.keras', 'keras', 'scaled', False),
    ('ANN SMOTE', MODELS_DIR + 'ann_bin_charging_smote_model.keras', 'keras', 'scaled', True),
    ('Stacking Classifier', MODELS_DIR + 'stacking_clf_bin_charging_best_model.pkl', 'sklearn', 'original', False),
]

# Load production results CSV
# Clean CSV columns for MLflow logging
df_prod = pd.read_csv('df_production_results_bin_charging.csv')

# Ensure numeric
for col in ['Prediction Time (sec)','F1 Macro', 'Memory Used (kB)', 'CPU Time (sec)', 'Physical Cores', 'Logical Cores', 'CPU % Used (Raw)',
            'CPU % Used (Normalized)','Model Size (MB)']:
    df_prod[col] = pd.to_numeric(df_prod[col], errors = 'coerce')

# Sanity checks
required_globals = ['X_train', 'X_test', 'X_test_scaled', 'Y_test_encoded', 'label_encoder']
missing = [g for g in required_globals if g not in globals()]
if missing:
    raise RuntimeError(f'Please ensure these globals exist before running: {missing}')

# Helper functions
def ensure_dataframe(X, ref_columns):
    if isinstance(X, np.ndarray):
        return pd.DataFrame(X, columns = list(ref_columns))
    if isinstance(X, pd.DataFrame):
        if list(X.columns) != list(ref_columns) and X.shape[1] == len(ref_columns):
            return pd.DataFrame(X.values, columns = list(ref_columns))
        return X
    raise ValueError('X must be numpy array or pandas DataFrame')

def model_includes_scaler(model):
    if isinstance(model, Pipeline):
        for step in model.named_steps.values():
            cls_name = step.__class__.__name__.lower()
            if 'scaler' in cls_name or 'standardscaler' in cls_name or 'minmax' in cls_name or 'preprocessor' in cls_name:
                return True
    return False

def _safe_params_dict(params: dict, prefix = ''):
    safe = {}
    for k, v in params.items():
        key = f'{prefix}{k}'
        if isinstance(v, (str, int, float, bool)) or v is None:
            safe[key] = v
        else:
            try:
                s = json.dumps(v) if isinstance(v, (dict, list)) else str(v)
                safe[key] = s[:200]
            except Exception:
                safe[key] = str(type(v))
    return safe

def keras_params_summary(model: keras.Model):
    params = {}
    try:
        params['keras_total_params'] = model.count_params()
    except Exception:
        params['keras_total_params'] = None
    params['keras_layers_count'] = len(model.layers)
    for i, layer in enumerate(model.layers):
        try:
            cfg = layer.get_config()
        except Exception:
            cfg = {}
        params[f'layer_{i}_class'] = layer.__class__.__name__
        if 'units' in cfg:
            params[f'layer_{i}_units'] = cfg.get('units')
        if 'activation' in cfg:
            params[f'layer_{i}_activation'] = cfg.get('activation')
    try:
        opt = model.optimizer.get_config()
        params['optimizer'] = model.optimizer.__class__.__name__
        for k, v in opt.items():
            params[f'opt_{k}'] = v if isinstance(v, (str, int, float, bool)) else str(v)
    except Exception:
        pass
    try:
        params['loss'] = model.loss if not callable(model.loss) else str(model.loss)
    except Exception:
        pass
    return params

def extract_xgb_from_pipeline(m):
    if not isinstance(m, Pipeline):
        return None
    for step in m.named_steps.values():
        cls = step.__class__.__name__.lower()
        if 'xgb' in cls or hasattr(step, 'get_xgb_params'):
            return step
    return None

def safe_head_example(X, n = 8):
    try:
        if isinstance(X, pd.DataFrame):
            return X.head(n)
        if isinstance(X, np.ndarray):
            return pd.DataFrame(X[:n], columns = X_train.columns)
    except Exception:
        pass
    return None

def safe_predict_for_signature(model, X_example, framework):
    try:
        fw = (framework or '').lower()
        if fw == 'keras':
            preds = model.predict(X_example, verbose = 0)
            return preds
        elif fw == 'xgboost':
            if isinstance(model, xgb.core.Booster):
                d = xgb.DMatrix(X_example)
                preds = model.predict(d)
                return preds
            else:
                if hasattr(model, 'predict'):
                    return model.predict(X_example)
        else:
            return model.predict(X_example)
    except Exception:
        return None

def safe_log_params_iter(obj, prefix = ''):
    safe = {}
    try:
        gp = obj.get_params()
        for k, v in gp.items():
            try:
                if isinstance(v, (list, dict)):
                    safe[f'{prefix}{k}'] = json.dumps(v)[:400]
                elif hasattr(v, '__name__') and getattr(v, '__name__'):
                    safe[f'{prefix}{k}'] = getattr(v, '__name__')
                else:
                    safe[f'{prefix}{k}'] = v
            except Exception:
                safe[f'{prefix}{k}'] = str(type(v))
    except Exception:
        pass
    return safe

def mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = None):
    fw = (framework or '').lower()
    example_df = safe_head_example(X_example) if X_example is not None else None

    input_example_np = None
    signature = None
    input_example = None

    if example_df is not None:
        input_example_np = example_df.values if isinstance(example_df, pd.DataFrame) else example_df
        input_example = input_example_np
        preds = safe_predict_for_signature(model, input_example_np, fw)
        try:
            signature = infer_signature(input_example_np, preds) if preds is not None else infer_signature(input_example_np, None)
        except Exception:
            signature = None

    with tempfile.TemporaryDirectory(prefix = 'mlflow_model_') as tmp_path:
        try:
            if fw == 'keras':
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                # Skip passing input_example here to avoid MLflow directory error
                try:
                    mlflow.keras.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception as e:
                    print('  -> Primary keras log_model failed:', e)
                    saved_path = os.path.join(tmp_path, 'model.keras')
                    os.makedirs(saved_path, exist_ok = True)
                    try:
                        model.save(saved_path, include_optimizer = True)
                    except Exception:
                        model.save(saved_path)
                    mlflow.log_artifact(saved_path, artifact_path = artifact_name)
                    return True

            elif fw == 'xgboost':
                inner = extract_xgb_from_pipeline(model) if isinstance(model, Pipeline) else None
                candidate = inner or model
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.xgboost.log_model(xgb_model = candidate, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                try:
                    if isinstance(candidate, xgb.core.Booster):
                        p = os.path.join(tmp_path, 'booster.json')
                        candidate.save_model(p)
                        mlflow.log_artifact(p, artifact_path = artifact_name)
                        return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

            else:
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.sklearn.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

        except Exception as e:
            print(f'mlflow_log_model_auto: primary path exception: {e}')
            return False

# Unified prediction/proba extractor (works for pipelines, sklearn, keras, xgboost)
def get_predictions_and_probas(model, framework, X_eval_df, label_encoder):
    """
    Return (y_pred_int_encoded, y_proba_for_auc_or_None)
    - y_pred: integer encoded (0/1 or 0..k-1)
    - y_proba_for_auc: 1D array of probability for positive class (binary) or numeric score usable by roc_auc_score
    """
    # Keras:
    if framework == 'keras':
        y_pred_proba = model.predict(X_eval_df, verbose=0)
        if y_pred_proba.ndim == 2 and y_pred_proba.shape[1] == 1:
            y_pred = (y_pred_proba.ravel() >= 0.5).astype(int)
            y_proba = y_pred_proba.ravel()
        else:
            # Multiclass softmax
            y_pred = np.argmax(y_pred_proba, axis = 1)
            # For multiclass AUC we might not compute here; for binary-case keep class 1 prob
            y_proba = y_pred_proba[:, 1] if y_pred_proba.shape[1] > 1 else y_pred_proba.ravel()
        return y_pred, y_proba

    # XGBoost: could be pipeline (with XGBClassifier inside), XGBClassifier object, or Booster saved
    if framework == 'xgboost':
        # If pipeline, pipeline.predict/predict_proba works
        try:
            # Prefer using predict_proba when exists
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_eval_df)
                if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                    y_pred = np.argmax(y_proba, axis = 1)
                    y_proba_for_auc = y_proba[:, 1]
                else:
                    y_pred = (y_proba.ravel() > 0.5).astype(int)
                    y_proba_for_auc = y_proba.ravel()
                return y_pred, y_proba_for_auc
            # Else maybe a Booster object loaded via xgb.Booster
            if isinstance(model, xgb.core.Booster):
                dmat = xgb.DMatrix(X_eval_df)
                probs = model.predict(dmat)
                # Binary or multiclass shape?
                if probs.ndim == 1:
                    y_pred = (probs > 0.5).astype(int)
                    return y_pred, probs.ravel()
                else:
                    # Multiclass
                    y_pred = np.argmax(probs, axis = 1)
                    # Choose class-1 probability for AUC if exists
                    y_proba_for_auc = probs[:, 1] if probs.shape[1] > 1 else probs.ravel()
                    return y_pred, y_proba_for_auc
        except Exception:
            pass

        # Last fallback: try predict
        y_pred_raw = model.predict(X_eval_df)
        if isinstance(y_pred_raw[0], str):
            y_pred = label_encoder.transform(y_pred_raw)
        else:
            y_pred = np.asarray(y_pred_raw).astype(int)
        # Try predict_proba but wrap in try
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                return y_pred, y_proba[:, 1]
            else:
                return y_pred, y_proba.ravel()
        except Exception:
            return y_pred, None

    # SKLearn generic (includes pipelines)
    y_pred_raw = model.predict(X_eval_df)
    if isinstance(y_pred_raw[0], str):
        y_pred = label_encoder.transform(y_pred_raw)
    else:
        y_pred = np.asarray(y_pred_raw).astype(int)

    # Try predict_proba
    y_proba_for_auc = None
    if hasattr(model, 'predict_proba'):
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                y_proba_for_auc = y_proba[:, 1]
            else:
                y_proba_for_auc = y_proba.ravel()
        except Exception:
            y_proba_for_auc = None
    elif hasattr(model, 'decision_function'):
        try:
            dec = model.decision_function(X_eval_df)
            # scale to [0,1]
            dec = np.ravel(dec)
            if dec.max() != dec.min():
                y_proba_for_auc = (dec - dec.min()) / (dec.max() - dec.min())
            else:
                y_proba_for_auc = dec
        except Exception:
            y_proba_for_auc = None

    return y_pred, y_proba_for_auc

# Main loop
f1_average = 'macro'
predictions_dict = {}

for model_name, path, framework, data_type, used_smote in models_to_log:
    print('\n' + '=' * 70)
    print(f'Loading {model_name}...')

    # ---------- Load model ----------
    model = None
    xgb_params_loaded = None
    try:
        if framework == 'keras':
            model = keras.models.load_model(path)
        elif framework == 'xgboost':
            # Attempt joblib (pipeline or sklearn wrapper)
            try:
                model = joblib.load(path)
            except Exception:
                # Attempt XGBClassifier .load_model()
                try:
                    clf = xgb.XGBClassifier()
                    clf.load_model(path)
                    model = clf
                except Exception:
                    # Attempt Booster
                    try:
                        booster = xgb.Booster()
                        booster.load_model(path)
                        model = booster
                    except Exception as e:
                        raise e
            # Companion params file if present
            params_path = path.replace('.json', '_params.json')
            if os.path.exists(params_path):
                try:
                    with open(params_path, 'r') as f:
                        xgb_params_loaded = json.load(f)
                except Exception:
                    xgb_params_loaded = None
        else:
            model = joblib.load(path)
    except Exception as e:
        print(f'Error loading {model_name}: {e}')
        continue

    # ---------- Choose evaluation data ----------
    use_original_X = False
    if model_name == 'Stacking Classifier':
        X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
        reason = "Stacking Classifier: Using original 'X_test' (trained that way)"
    else:
        if framework != 'keras' and model_includes_scaler(model):
            X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
            reason = "Pipeline includes scaler: Using original 'X_test' (pipeline will scale)"
        else:
            X_eval = X_test_scaled if data_type == 'scaled' else X_test
            X_eval_df = ensure_dataframe(X_eval, ref_columns = X_train.columns)
            reason = f"Data type = '{data_type}': Using '{'X_test_scaled' if data_type == 'scaled' else 'X_test'}'"

    print('  ->', reason)

    # ---------- Predictions ----------
    try:
        y_pred, y_proba_for_auc = get_predictions_and_probas(model, framework, X_eval_df, label_encoder)
    except Exception as e:
        print(f'Prediction failed for {model_name}: {e}')
        continue

    # ---------- Metrics ----------
    y_true_enc = Y_test_encoded
    acc = accuracy_score(y_true_enc, y_pred)
    f1 = f1_score(y_true_enc, y_pred, average = f1_average)
    auc_val = None
    if y_proba_for_auc is not None:
        try:
            auc_val = roc_auc_score(y_true_enc, y_proba_for_auc)
        except Exception as e:
            print(f'  -> Warning: roc_auc_score failed: {e}; y_proba shape: {np.shape(y_proba_for_auc)}')

    # Print the metrics
    print(f'  -> Accuracy: {acc:.4f}, F1 ({f1_average}): {f1:.4f}, ROC AUC: {auc_val:.4f}')
    print('  -> Sample preds (first 10):', list(y_pred[:10]))
    unique, counts = np.unique(y_pred, return_counts = True)
    print('  -> Unique pred counts:', dict(zip(unique.tolist(), counts.tolist())))

    predictions_dict[model_name] = y_pred

    # ---------- MLflow logging ----------
    run_name = f'{model_name} Binary (Charging State)'
    with mlflow.start_run(run_name = run_name) as run:
        row = df_prod[df_prod['Model'] == model_name]
        if not row.empty:
            mlflow.log_metric('Prediction Time secs', float(row['Prediction Time (sec)'].iloc[0]))
          # mlflow.log_metric('Memory Used (kB)', float(row['Memory Used (kB)'].iloc[0]))
            mlflow.log_metric('CPU Time secs', float(row['CPU Time (sec)'].iloc[0]))
            mlflow.log_metric('Physical Cores', int(row['Physical Cores'].iloc[0]))
            mlflow.log_metric('Logical Cores', int(row['Logical Cores'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Raw', float(row['CPU % Used (Raw)'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Normalized', float(row['CPU % Used (Normalized)'].iloc[0]))
            mlflow.log_metric('Model Size MB', float(row['Model Size (MB)'].iloc[0]))
          # mlflow.log_metric('F1 Macro (CSV)', float(row['F1 Macro'])) # I log it from the model not the csv

        mlflow.log_metric('Accuracy', float(f'{acc:.4f}'))
        mlflow.log_metric('F1 macro', float(f'{f1:.4f}'))
        if auc_val is not None:
            mlflow.log_metric('ROC AUC', float(f'{auc_val:.4f}'))

        mlflow.log_param('SMOTE', bool(used_smote))
        mlflow.set_tag('SMOTE', str(bool(used_smote)))
        mlflow.log_param('Framework', framework)
        mlflow.log_param('ModelPath', path)

        # ---------- Log model parameters robustly ----------
        try:
            if framework == 'keras':
                kp = keras_params_summary(model)
                safe_kp = _safe_params_dict(kp, prefix = 'keras_')
                if safe_kp:
                    mlflow.log_params(safe_kp)

            elif framework == 'xgboost':
                if isinstance(model, Pipeline):
                    inner = extract_xgb_from_pipeline(model)
                    if inner:
                        try:
                            safe = safe_log_params_iter(inner, prefix = 'xgb_')
                            if safe:
                                mlflow.log_params(safe)
                        except Exception:
                            pass
                else:
                    try:
                        if hasattr(model, 'get_xgb_params'):
                            xp = model.get_xgb_params()
                            mlflow.log_params(_safe_params_dict(xp, prefix = 'xgb_'))
                    except Exception:
                        pass
                    try:
                        if hasattr(model, 'get_params'):
                            gp = model.get_params()
                            safe_gp = _safe_params_dict(gp, prefix = 'xgb_')
                            if safe_gp:
                                mlflow.log_params(safe_gp)
                    except Exception:
                        pass
                    if xgb_params_loaded:
                        mlflow.log_params(_safe_params_dict(xgb_params_loaded, prefix = 'xgb_file_'))

            elif 'stack' in model_name.lower() or hasattr(model, 'final_estimator_'):
                try:
                    # Meta estimator
                    if hasattr(model, 'final_estimator_'):
                        try:
                            meta_params = safe_log_params_iter(model.final_estimator_, prefix = 'meta_')
                            if meta_params:
                                mlflow.log_params(meta_params)
                        except Exception:
                            pass
                    # Base estimators: named_estimators_ or estimators_
                    if hasattr(model, 'named_estimators_'):
                        for name, est in model.named_estimators_.items():
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{name}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                # Skip problematic estimator param
                                pass
                    elif hasattr(model, 'estimators_'):
                        for i, est in enumerate(model.estimators_):
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{i}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                pass
                except Exception as e:
                    print('  -> Warning logging stacking params:', e)

            else:
                if hasattr(model, 'get_params'):
                    try:
                        gp = model.get_params()
                        mlflow.log_params(_safe_params_dict(gp))
                    except Exception:
                        if isinstance(model, Pipeline):
                            for name, step in model.named_steps.items():
                                if hasattr(step, 'get_params'):
                                    mlflow.log_params(_safe_params_dict(step.get_params(), prefix = f'{name}__'))
        except Exception as e:
            print('  -> Warning while preparing params to log:', e)

        # ---------- Classification report: safe JSON + text ----------
        try:
            ncls = len(np.unique(y_true_enc))
            if 'label_encoder' in globals() and len(label_encoder.classes_) == ncls:
                target_names = [str(s) for s in label_encoder.classes_]
            else:
                target_names = [str(int(x)) for x in np.unique(y_true_enc)]

            creport_dict = classification_report(y_true_enc, y_pred, output_dict = True)
            creport_text = classification_report(y_true_enc, y_pred, target_names = target_names)
            mlflow.log_text(json.dumps(creport_dict, indent = 4), 'classification_report.json')
            mlflow.log_text(creport_text, 'classification_report.txt')
        except Exception as e:
            print('  -> Warning logging classification report:', e)

        # ---------- Log model artifact robustly (with signatures & input_example) ----------
        try:
            logged_ok = mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = X_eval_df)
            if not logged_ok:
                print('  -> Warning: model artifact not logged via standard mlflow path and fallback.')
        except Exception as e:
            print('  -> Error logging model artifact:', e)

        # ---------- Print run link/info ----------
        try:
            run_id = run.info.run_id
            exp_id = run.info.experiment_id
            print(f'  -> MLflow run id: {run_id} (experiment {exp_id})')
        except Exception:
            pass

    print(f'✅ Logged {model_name} (SMOTE = {used_smote})')

# After loop: cross-model identical prediction check
print('\n' + '=' * 70)
print('Cross-model prediction overlap:')
names = list(predictions_dict.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = names[i], names[j]
        overlap = np.mean(predictions_dict[a] == predictions_dict[b])
        print(f'  - {a} vs {b}: identical fraction = {overlap:.3f}')


Loading KNN...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.8461, F1 (macro): 0.7720, ROC AUC: 0.8828
  -> Sample preds (first 10): [1, 0, 1, 1, 1, 0, 1, 0, 1, 1]
  -> Unique pred counts: {0: 1156, 1: 4763}
  -> MLflow run id: e12a6d9407844bee91d5f48b4318dd77 (experiment 3)
🏃 View run KNN Binary (Charging State) at: http://127.0.0.1:5000/#/experiments/3/runs/e12a6d9407844bee91d5f48b4318dd77
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
✅ Logged KNN (SMOTE = False)

Loading KNN SMOTE...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.8104, F1 (macro): 0.7649, ROC AUC: 0.8775
  -> Sample preds (first 10): [1, 0, 1, 0, 0, 0, 1, 0, 1, 1]
  -> Unique pred counts: {0: 1925, 1: 3994}
  -> MLflow run id: a1188a16051347cab216dcffd08a54ed (experiment 3)
🏃 View run KNN SMOTE Binary (Charging State) at: http://127.0.0.1:5000/#/experiments/3/runs/a1188a16051347cab216dcffd08a54ed
🧪 View expe

# Binary Classification (Both States)

In [37]:
# Set the experiment in MLflow
mlflow.set_experiment('Binary Classification (Both States)')

<Experiment: artifact_location='/Users/yannis/mlflow-artifacts/4', creation_time=1761300495818, experiment_id='4', last_update_time=1761300495818, lifecycle_stage='active', name='Binary Classification (Both States)', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [38]:
# Load the sheet into a DataFrame
df = pd.read_csv(DATA_PATH)

# Display basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115298 entries, 0 to 115297
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   time           115298 non-null  object 
 1   shunt_voltage  115298 non-null  int64  
 2   bus_voltage_V  115298 non-null  float64
 3   current_mA     115298 non-null  int64  
 4   power_mW       115298 non-null  int64  
 5   State          115298 non-null  object 
 6   Attack         115298 non-null  object 
 7   Attack-Group   115298 non-null  object 
 8   Label          115298 non-null  object 
 9   interface      115298 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 8.8+ MB


In [39]:
# Define mapping dictionary
attack_mapping = {
    'syn-flood': 'DoS',
    'tcp-flood': 'DoS',
    'cryptojacking': 'Cryptojacking',
    'vuln-scan': 'Recon',
    'syn-stealth': 'Recon',
    'Backdoor': 'Backdoor',
    'none': 'Benign'
}

# Apply mapping to create 'Attack-New' column
df['Attack-New'] = df['Attack'].map(attack_mapping)

# Print the first rows of the DataFrame to verify the new column
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp,DoS
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp,DoS
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp,DoS
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp,DoS
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp,DoS


In [40]:
# Convert "Attack-New" to binary: 1 = Attack, 0 = Benign
df['Attack-Binary'] = df['Attack-New'].apply(lambda x: 0 if x == 'Benign' else 1)

# Print the first few rows of the DataFrame to verify the new column
df.head()

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface,Attack-New,Attack-Binary
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,DoS,attack,ocpp,DoS,1
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,DoS,attack,ocpp,DoS,1
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,DoS,attack,ocpp,DoS,1
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,DoS,attack,ocpp,DoS,1
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,DoS,attack,ocpp,DoS,1


In [41]:
# Select numerical features for classification
numerical_features = ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']

# Define features (X) and target variable (Y)
X = df[numerical_features]
Y = df['Attack-Binary']

In [42]:
# Split the dataset into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42, stratify = Y, shuffle = True)

In [43]:
# Split the training set into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.2, random_state = 42, stratify = Y_train, shuffle = True)

In [44]:
# Set the encoder 
label_encoder = LabelEncoder()

# Encode 'Attack-New' labels as numbers for each subset
# Note: This is done after splitting to avoid data leakage
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_val_encoded = label_encoder.transform(Y_val)
Y_test_encoded = label_encoder.transform(Y_test)

# Check what each number represents
print('Label encoder classes:', label_encoder.classes_)

Label encoder classes: [0 1]


In [45]:
# Create the scaler
scaler = StandardScaler()

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Shape of the scaled data
print(f'Shape of X_train_scaled: {X_train_scaled.shape}')
print(f'Shape of X_val_scaled: {X_val_scaled.shape}')
print(f'Shape of X_test_scaled: {X_test_scaled.shape}')

Shape of X_train_scaled: (73790, 4)
Shape of X_val_scaled: (18448, 4)
Shape of X_test_scaled: (23060, 4)


In [ ]:
# Suppress specific warnings for cleaner output
warnings.filterwarnings('ignore', category = UserWarning)

# Models list
models_to_log = [
    ('KNN', MODELS_DIR + 'knn_bin_both_best_model.pkl', 'sklearn', 'scaled', False),
    ('KNN SMOTE', MODELS_DIR + 'knn_bin_both_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('XGBoost', MODELS_DIR + 'xgboost_bin_both_best_model.json', 'xgboost', 'scaled', False),
    ('XGBoost SMOTE', MODELS_DIR + 'xgboost_bin_both_smote_best_model.json', 'xgboost', 'scaled', True),
    ('Logistic Regression', MODELS_DIR + 'logistic_regression_bin_both_best_model.pkl', 'sklearn', 'scaled', False),
    ('Logistic Regression SMOTE', MODELS_DIR + 'logistic_regression_bin_both_smote_best_model.pkl', 'sklearn', 'scaled', True),
    ('Random Forest', MODELS_DIR + 'random_forest_bin_both_best_model.pkl', 'sklearn', 'original', False),
    ('Random Forest SMOTE', MODELS_DIR + 'random_forest_bin_both_smote_best_model.pkl', 'sklearn', 'original', True),
    ('ANN', MODELS_DIR + 'ann_bin_both_model.keras', 'keras', 'scaled', False),
    ('ANN Weighted', MODELS_DIR + 'ann_bin_both_weighted_model.keras', 'keras', 'scaled', False),
    ('ANN SMOTE', MODELS_DIR + 'ann_bin_both_smote_model.keras', 'keras', 'scaled', True),
    ('Stacking Classifier', MODELS_DIR + 'stacking_clf_bin_both_best_model.pkl', 'sklearn', 'original', False),
]

# Load production results CSV
# Clean CSV columns for MLflow logging
df_prod = pd.read_csv('df_production_results_bin_both.csv')

# Ensure numeric
for col in ['Prediction Time (sec)','F1 Macro', 'Memory Used (kB)', 'CPU Time (sec)', 'Physical Cores', 'Logical Cores', 'CPU % Used (Raw)',
            'CPU % Used (Normalized)','Model Size (MB)']:
    df_prod[col] = pd.to_numeric(df_prod[col], errors = 'coerce')

# Sanity checks
required_globals = ['X_train', 'X_test', 'X_test_scaled', 'Y_test_encoded', 'label_encoder']
missing = [g for g in required_globals if g not in globals()]
if missing:
    raise RuntimeError(f'Please ensure these globals exist before running: {missing}')

# Helper functions
def ensure_dataframe(X, ref_columns):
    if isinstance(X, np.ndarray):
        return pd.DataFrame(X, columns = list(ref_columns))
    if isinstance(X, pd.DataFrame):
        if list(X.columns) != list(ref_columns) and X.shape[1] == len(ref_columns):
            return pd.DataFrame(X.values, columns = list(ref_columns))
        return X
    raise ValueError('X must be numpy array or pandas DataFrame')

def model_includes_scaler(model):
    if isinstance(model, Pipeline):
        for step in model.named_steps.values():
            cls_name = step.__class__.__name__.lower()
            if 'scaler' in cls_name or 'standardscaler' in cls_name or 'minmax' in cls_name or 'preprocessor' in cls_name:
                return True
    return False

def _safe_params_dict(params: dict, prefix = ''):
    safe = {}
    for k, v in params.items():
        key = f'{prefix}{k}'
        if isinstance(v, (str, int, float, bool)) or v is None:
            safe[key] = v
        else:
            try:
                s = json.dumps(v) if isinstance(v, (dict, list)) else str(v)
                safe[key] = s[:200]
            except Exception:
                safe[key] = str(type(v))
    return safe

def keras_params_summary(model: keras.Model):
    params = {}
    try:
        params['keras_total_params'] = model.count_params()
    except Exception:
        params['keras_total_params'] = None
    params['keras_layers_count'] = len(model.layers)
    for i, layer in enumerate(model.layers):
        try:
            cfg = layer.get_config()
        except Exception:
            cfg = {}
        params[f'layer_{i}_class'] = layer.__class__.__name__
        if 'units' in cfg:
            params[f'layer_{i}_units'] = cfg.get('units')
        if 'activation' in cfg:
            params[f'layer_{i}_activation'] = cfg.get('activation')
    try:
        opt = model.optimizer.get_config()
        params['optimizer'] = model.optimizer.__class__.__name__
        for k, v in opt.items():
            params[f'opt_{k}'] = v if isinstance(v, (str, int, float, bool)) else str(v)
    except Exception:
        pass
    try:
        params['loss'] = model.loss if not callable(model.loss) else str(model.loss)
    except Exception:
        pass
    return params

def extract_xgb_from_pipeline(m):
    if not isinstance(m, Pipeline):
        return None
    for step in m.named_steps.values():
        cls = step.__class__.__name__.lower()
        if 'xgb' in cls or hasattr(step, 'get_xgb_params'):
            return step
    return None

def safe_head_example(X, n=8):
    try:
        if isinstance(X, pd.DataFrame):
            return X.head(n)
        if isinstance(X, np.ndarray):
            return pd.DataFrame(X[:n], columns = X_train.columns)
    except Exception:
        pass
    return None

def safe_predict_for_signature(model, X_example, framework):
    try:
        fw = (framework or "").lower()
        if fw == 'keras':
            preds = model.predict(X_example, verbose = 0)
            return preds
        elif fw == 'xgboost':
            if isinstance(model, xgb.core.Booster):
                d = xgb.DMatrix(X_example)
                preds = model.predict(d)
                return preds
            else:
                if hasattr(model, 'predict'):
                    return model.predict(X_example)
        else:
            return model.predict(X_example)
    except Exception:
        return None

def safe_log_params_iter(obj, prefix = ''):
    safe = {}
    try:
        gp = obj.get_params()
        for k, v in gp.items():
            try:
                if isinstance(v, (list, dict)):
                    safe[f'{prefix}{k}'] = json.dumps(v)[:400]
                elif hasattr(v, '__name__') and getattr(v, '__name__'):
                    safe[f'{prefix}{k}'] = getattr(v, '__name__')
                else:
                    safe[f'{prefix}{k}'] = v
            except Exception:
                safe[f'{prefix}{k}'] = str(type(v))
    except Exception:
        pass
    return safe

def mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = None):
    fw = (framework or '').lower()
    example_df = safe_head_example(X_example) if X_example is not None else None

    input_example_np = None
    signature = None
    input_example = None

    if example_df is not None:
        input_example_np = example_df.values if isinstance(example_df, pd.DataFrame) else example_df
        input_example = input_example_np
        preds = safe_predict_for_signature(model, input_example_np, fw)
        try:
            signature = infer_signature(input_example_np, preds) if preds is not None else infer_signature(input_example_np, None)
        except Exception:
            signature = None

    with tempfile.TemporaryDirectory(prefix = 'mlflow_model_') as tmp_path:
        try:
            if fw == 'keras':
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                # Skip passing input_example here to avoid MLflow directory error
                try:
                    mlflow.keras.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception as e:
                    print('  -> Primary keras log_model failed:', e)
                    saved_path = os.path.join(tmp_path, 'model.keras')
                    os.makedirs(saved_path, exist_ok = True)
                    try:
                        model.save(saved_path, include_optimizer = True)
                    except Exception:
                        model.save(saved_path)
                    mlflow.log_artifact(saved_path, artifact_path = artifact_name)
                    return True

            elif fw == 'xgboost':
                inner = extract_xgb_from_pipeline(model) if isinstance(model, Pipeline) else None
                candidate = inner or model
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.xgboost.log_model(xgb_model = candidate, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                try:
                    if isinstance(candidate, xgb.core.Booster):
                        p = os.path.join(tmp_path, 'booster.json')
                        candidate.save_model(p)
                        mlflow.log_artifact(p, artifact_path = artifact_name)
                        return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

            else:
                kwargs = {}
                if signature is not None:
                    kwargs['signature'] = signature
                if input_example is not None:
                    kwargs['input_example'] = input_example
                try:
                    mlflow.sklearn.log_model(model, name = artifact_name, **kwargs)
                    return True
                except Exception:
                    pass
                pkl = os.path.join(tmp_path, 'model_joblib.pkl')
                joblib.dump(model, pkl)
                mlflow.log_artifact(pkl, artifact_path = artifact_name)
                return True

        except Exception as e:
            print(f'mlflow_log_model_auto: primary path exception: {e}')
            return False

# Unified prediction/proba extractor (works for pipelines, sklearn, keras, xgboost)
def get_predictions_and_probas(model, framework, X_eval_df, label_encoder):
    """
    Return (y_pred_int_encoded, y_proba_for_auc_or_None)
    - y_pred: integer encoded (0/1 or 0..k-1)
    - y_proba_for_auc: 1D array of probability for positive class (binary) or numeric score usable by roc_auc_score
    """
    # Keras:
    if framework == 'keras':
        y_pred_proba = model.predict(X_eval_df, verbose = 0)
        if y_pred_proba.ndim == 2 and y_pred_proba.shape[1] == 1:
            y_pred = (y_pred_proba.ravel() >= 0.5).astype(int)
            y_proba = y_pred_proba.ravel()
        else:
            # Multiclass softmax
            y_pred = np.argmax(y_pred_proba, axis = 1)
            # For multiclass AUC we might not compute here; for binary-case keep class 1 prob
            y_proba = y_pred_proba[:, 1] if y_pred_proba.shape[1] > 1 else y_pred_proba.ravel()
        return y_pred, y_proba

    # XGBoost: could be pipeline (with XGBClassifier inside), XGBClassifier object, or Booster saved
    if framework == 'xgboost':
        # If pipeline, pipeline.predict/predict_proba works
        try:
            # Prefer using predict_proba when exists
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_eval_df)
                if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                    y_pred = np.argmax(y_proba, axis=1)
                    y_proba_for_auc = y_proba[:, 1]
                else:
                    y_pred = (y_proba.ravel() > 0.5).astype(int)
                    y_proba_for_auc = y_proba.ravel()
                return y_pred, y_proba_for_auc
            # Else maybe a Booster object loaded via xgb.Booster
            if isinstance(model, xgb.core.Booster):
                dmat = xgb.DMatrix(X_eval_df)
                probs = model.predict(dmat)
                # Binary or multiclass shape?
                if probs.ndim == 1:
                    y_pred = (probs > 0.5).astype(int)
                    return y_pred, probs.ravel()
                else:
                    # Multiclass
                    y_pred = np.argmax(probs, axis = 1)
                    # Choose class-1 probability for AUC if exists
                    y_proba_for_auc = probs[:, 1] if probs.shape[1] > 1 else probs.ravel()
                    return y_pred, y_proba_for_auc
        except Exception:
            pass

        # Last fallback: try predict
        y_pred_raw = model.predict(X_eval_df)
        if isinstance(y_pred_raw[0], str):
            y_pred = label_encoder.transform(y_pred_raw)
        else:
            y_pred = np.asarray(y_pred_raw).astype(int)
        # Try predict_proba but wrap in try
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                return y_pred, y_proba[:, 1]
            else:
                return y_pred, y_proba.ravel()
        except Exception:
            return y_pred, None

    # SKLearn generic (includes pipelines)
    y_pred_raw = model.predict(X_eval_df)
    if isinstance(y_pred_raw[0], str):
        y_pred = label_encoder.transform(y_pred_raw)
    else:
        y_pred = np.asarray(y_pred_raw).astype(int)

    # Try predict_proba
    y_proba_for_auc = None
    if hasattr(model, 'predict_proba'):
        try:
            y_proba = model.predict_proba(X_eval_df)
            if y_proba.ndim == 2 and y_proba.shape[1] >= 2:
                y_proba_for_auc = y_proba[:, 1]
            else:
                y_proba_for_auc = y_proba.ravel()
        except Exception:
            y_proba_for_auc = None
    elif hasattr(model, 'decision_function'):
        try:
            dec = model.decision_function(X_eval_df)
            # scale to [0,1]
            dec = np.ravel(dec)
            if dec.max() != dec.min():
                y_proba_for_auc = (dec - dec.min()) / (dec.max() - dec.min())
            else:
                y_proba_for_auc = dec
        except Exception:
            y_proba_for_auc = None

    return y_pred, y_proba_for_auc

# Main loop
f1_average = 'macro'
predictions_dict = {}

for model_name, path, framework, data_type, used_smote in models_to_log:
    print('\n' + '=' * 70)
    print(f'Loading {model_name}...')

    # ---------- Load model ----------
    model = None
    xgb_params_loaded = None
    try:
        if framework == 'keras':
            model = keras.models.load_model(path)
        elif framework == 'xgboost':
            # Attempt joblib (pipeline or sklearn wrapper)
            try:
                model = joblib.load(path)
            except Exception:
                # Attempt XGBClassifier .load_model()
                try:
                    clf = xgb.XGBClassifier()
                    clf.load_model(path)
                    model = clf
                except Exception:
                    # Attempt Booster
                    try:
                        booster = xgb.Booster()
                        booster.load_model(path)
                        model = booster
                    except Exception as e:
                        raise e
            # Companion params file if present
            params_path = path.replace('.json', '_params.json')
            if os.path.exists(params_path):
                try:
                    with open(params_path, 'r') as f:
                        xgb_params_loaded = json.load(f)
                except Exception:
                    xgb_params_loaded = None
        else:
            model = joblib.load(path)
    except Exception as e:
        print(f'Error loading {model_name}: {e}')
        continue

    # ---------- Choose evaluation data ----------
    use_original_X = False
    if model_name == 'Stacking Classifier':
        X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
        reason = "Stacking Classifier: Using original 'X_test' (trained that way)"
    else:
        if framework != 'keras' and model_includes_scaler(model):
            X_eval_df = ensure_dataframe(X_test, ref_columns = X_train.columns)
            reason = "Pipeline includes scaler: Using original 'X_test' (pipeline will scale)"
        else:
            X_eval = X_test_scaled if data_type == 'scaled' else X_test
            X_eval_df = ensure_dataframe(X_eval, ref_columns = X_train.columns)
            reason = f"Data type = '{data_type}': Using '{'X_test_scaled' if data_type == 'scaled' else 'X_test'}'"

    print('  ->', reason)

    # ---------- Predictions ----------
    try:
        y_pred, y_proba_for_auc = get_predictions_and_probas(model, framework, X_eval_df, label_encoder)
    except Exception as e:
        print(f'Prediction failed for {model_name}: {e}')
        continue

    # ---------- Metrics ----------
    y_true_enc = Y_test_encoded
    acc = accuracy_score(y_true_enc, y_pred)
    f1 = f1_score(y_true_enc, y_pred, average = f1_average)
    auc_val = None
    if y_proba_for_auc is not None:
        try:
            auc_val = roc_auc_score(y_true_enc, y_proba_for_auc)
        except Exception as e:
            print(f'  -> Warning: roc_auc_score failed: {e}; y_proba shape: {np.shape(y_proba_for_auc)}')

    # Print the metrics
    print(f'  -> Accuracy: {acc:.4f}, F1 ({f1_average}): {f1:.4f}, ROC AUC: {auc_val:.4f}')
    print('  -> Sample preds (first 10):', list(y_pred[:10]))
    unique, counts = np.unique(y_pred, return_counts = True)
    print('  -> Unique pred counts:', dict(zip(unique.tolist(), counts.tolist())))

    predictions_dict[model_name] = y_pred

    # ---------- MLflow logging ----------
    run_name = f'{model_name} Binary (Both States)'
    with mlflow.start_run(run_name = run_name) as run:
        row = df_prod[df_prod['Model'] == model_name]
        if not row.empty:
            mlflow.log_metric('Prediction Time secs', float(row['Prediction Time (sec)'].iloc[0]))
          # mlflow.log_metric('Memory Used (kB)', float(row['Memory Used (kB)'].iloc[0]))
            mlflow.log_metric('CPU Time secs', float(row['CPU Time (sec)'].iloc[0]))
            mlflow.log_metric('Physical Cores', int(row['Physical Cores'].iloc[0]))
            mlflow.log_metric('Logical Cores', int(row['Logical Cores'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Raw', float(row['CPU % Used (Raw)'].iloc[0]))
            mlflow.log_metric('CPU Percentage Used Normalized', float(row['CPU % Used (Normalized)'].iloc[0]))
            mlflow.log_metric('Model Size MB', float(row['Model Size (MB)'].iloc[0]))
          # mlflow.log_metric('F1 Macro (CSV)', float(row['F1 Macro'])) # I log it from the model not the csv
          
        mlflow.log_metric('Accuracy', float(f'{acc:.4f}'))
        mlflow.log_metric('F1 macro', float(f'{f1:.4f}'))
        if auc_val is not None:
            mlflow.log_metric('ROC AUC', float(f'{auc_val:.4f}'))

        mlflow.log_param('SMOTE', bool(used_smote))
        mlflow.set_tag('SMOTE', str(bool(used_smote)))
        mlflow.log_param('Framework', framework)
        mlflow.log_param('ModelPath', path)

        # ---------- Log model parameters robustly ----------
        try:
            if framework == 'keras':
                kp = keras_params_summary(model)
                safe_kp = _safe_params_dict(kp, prefix = 'keras_')
                if safe_kp:
                    mlflow.log_params(safe_kp)

            elif framework == 'xgboost':
                if isinstance(model, Pipeline):
                    inner = extract_xgb_from_pipeline(model)
                    if inner:
                        try:
                            safe = safe_log_params_iter(inner, prefix = 'xgb_')
                            if safe:
                                mlflow.log_params(safe)
                        except Exception:
                            pass
                else:
                    try:
                        if hasattr(model, 'get_xgb_params'):
                            xp = model.get_xgb_params()
                            mlflow.log_params(_safe_params_dict(xp, prefix = 'xgb_'))
                    except Exception:
                        pass
                    try:
                        if hasattr(model, 'get_params'):
                            gp = model.get_params()
                            safe_gp = _safe_params_dict(gp, prefix = 'xgb_')
                            if safe_gp:
                                mlflow.log_params(safe_gp)
                    except Exception:
                        pass
                    if xgb_params_loaded:
                        mlflow.log_params(_safe_params_dict(xgb_params_loaded, prefix = 'xgb_file_'))

            elif 'stack' in model_name.lower() or hasattr(model, 'final_estimator_'):
                try:
                    # Meta estimator
                    if hasattr(model, 'final_estimator_'):
                        try:
                            meta_params = safe_log_params_iter(model.final_estimator_, prefix = 'meta_')
                            if meta_params:
                                mlflow.log_params(meta_params)
                        except Exception:
                            pass
                    # Base estimators: named_estimators_ or estimators_
                    if hasattr(model, 'named_estimators_'):
                        for name, est in model.named_estimators_.items():
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{name}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                # Skip problematic estimator param
                                pass
                    elif hasattr(model, 'estimators_'):
                        for i, est in enumerate(model.estimators_):
                            try:
                                est_params = safe_log_params_iter(est, prefix = f'base_{i}_')
                                if est_params:
                                    mlflow.log_params(est_params)
                            except Exception:
                                pass
                except Exception as e:
                    print('  -> Warning logging stacking params:', e)

            else:
                if hasattr(model, 'get_params'):
                    try:
                        gp = model.get_params()
                        mlflow.log_params(_safe_params_dict(gp))
                    except Exception:
                        if isinstance(model, Pipeline):
                            for name, step in model.named_steps.items():
                                if hasattr(step, 'get_params'):
                                    mlflow.log_params(_safe_params_dict(step.get_params(), prefix = f'{name}__'))
        except Exception as e:
            print('  -> Warning while preparing params to log:', e)

        # ---------- Classification report: safe JSON + text ----------
        try:
            ncls = len(np.unique(y_true_enc))
            if 'label_encoder' in globals() and len(label_encoder.classes_) == ncls:
                target_names = [str(s) for s in label_encoder.classes_]
            else:
                target_names = [str(int(x)) for x in np.unique(y_true_enc)]

            creport_dict = classification_report(y_true_enc, y_pred, output_dict = True)
            creport_text = classification_report(y_true_enc, y_pred, target_names = target_names)
            mlflow.log_text(json.dumps(creport_dict, indent = 4), 'classification_report.json')
            mlflow.log_text(creport_text, 'classification_report.txt')
        except Exception as e:
            print('  -> Warning logging classification report:', e)

        # ---------- Log model artifact robustly (with signatures & input_example) ----------
        try:
            logged_ok = mlflow_log_model_auto(model, framework, artifact_name = 'model', X_example = X_eval_df)
            if not logged_ok:
                print('  -> Warning: model artifact not logged via standard mlflow path and fallback.')
        except Exception as e:
            print('  -> Error logging model artifact:', e)

        # ---------- Print run link/info ----------
        try:
            run_id = run.info.run_id
            exp_id = run.info.experiment_id
            print(f'  -> MLflow run id: {run_id} (experiment {exp_id})')
        except Exception:
            pass

    print(f'✅ Logged {model_name} (SMOTE = {used_smote})')

# After loop: cross-model identical prediction check
print('\n' + '=' * 70)
print('Cross-model prediction overlap:')
names = list(predictions_dict.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = names[i], names[j]
        overlap = np.mean(predictions_dict[a] == predictions_dict[b])
        print(f'  - {a} vs {b}: identical fraction = {overlap:.3f}')


Loading KNN...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.9378, F1 (macro): 0.8484, ROC AUC: 0.9432
  -> Sample preds (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  -> Unique pred counts: {0: 2480, 1: 20580}
  -> MLflow run id: f021580d70c44f33bc0c6946f21699e5 (experiment 4)
🏃 View run KNN Binary (Both States) at: http://127.0.0.1:5000/#/experiments/4/runs/f021580d70c44f33bc0c6946f21699e5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
✅ Logged KNN (SMOTE = False)

Loading KNN SMOTE...
  -> Pipeline includes scaler: Using original 'X_test' (pipeline will scale)
  -> Accuracy: 0.9058, F1 (macro): 0.8164, ROC AUC: 0.9286
  -> Sample preds (first 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  -> Unique pred counts: {0: 4098, 1: 18962}
  -> MLflow run id: c696a970ca914294b7dc0e07b0b08529 (experiment 4)
🏃 View run KNN SMOTE Binary (Both States) at: http://127.0.0.1:5000/#/experiments/4/runs/c696a970ca914294b7dc0e07b0b08529
🧪 View experime